---
## **Residual MAX**

##### **데이터 준비**

In [1]:
import pandas as pd

# Investment Factor 계산에 필요한 데이터를 input 폴더에서 불러옴
excess_return    = pd.read_csv("./input/KOSPI_KOSDAQ_excess_return (251031-251130).csv", index_col=0, parse_dates=True)     # 초과수익률 (종속변수)
factors          = pd.read_csv("./input/factors_daily (251031-251130).csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)

In [2]:
# 수익률 분해를 위한 팩터 정리 - 독립변수 설정 (FF3 팩터만 사용)
factors['MKT'] = factors['KOSPI'] - factors['RF']            # MKT 팩터(시장 포트폴리오 초과 수익률)
X = factors[['MKT', 'SMB', 'HML']].copy()

##### **기간 설정 (테스트용)**
# 기간을 설정하여 일부 데이터만 사용할 수 있습니다
# None으로 설정하면 전체 기간을 사용합니다
START_DATE = '2025-10-31'  # 예: '2024-01-01'
END_DATE = '2025-11-30'    # 예: '2024-12-31'

# 데이터 필터링
if START_DATE is not None or END_DATE is not None:
    if START_DATE is not None:
        excess_return = excess_return.loc[excess_return.index >= START_DATE]
        X = X.loc[X.index >= START_DATE]
    if END_DATE is not None:
        excess_return = excess_return.loc[excess_return.index <= END_DATE]
        X = X.loc[X.index <= END_DATE]
    
    print(f"데이터 기간: {excess_return.index.min()} ~ {excess_return.index.max()}")
    print(f"총 거래일 수: {len(excess_return)}")
else:
    print(f"전체 기간 사용: {excess_return.index.min()} ~ {excess_return.index.max()}")
    print(f"총 거래일 수: {len(excess_return)}")

데이터 기간: 2025-10-31 00:00:00 ~ 2025-11-28 00:00:00
총 거래일 수: 21


##### **함수 정의**

In [4]:
import statsmodels.api as sm
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np
import os

# output/FF3 폴더에 MAX1.csv, MAX5.csv 파일이 있는지 확인
output_dir = './output/FF3'
max1_file = os.path.join(output_dir, 'MAX1 (251130).csv')
max5_file = os.path.join(output_dir, 'MAX5 (251130).csv')

if os.path.exists(max1_file) and os.path.exists(max5_file):
    # 파일이 있으면 잔차 계산을 건너뛰고 파일을 읽어옴
    print(f"기존 파일이 존재합니다. 잔차 계산을 건너뜁니다.")
    print(f"MAX1.csv 로드 중: {max1_file}")
    MAX1_df = pd.read_csv(max1_file, index_col=0, parse_dates=True)
    print(f"MAX5.csv 로드 중: {max5_file}")
    MAX5_df = pd.read_csv(max5_file, index_col=0, parse_dates=True)
    print("기존 MAX1, MAX5 데이터 로드 완료!")
    print(f"MAX1 shape: {MAX1_df.shape}")
    print(f"MAX5 shape: {MAX5_df.shape}")
    print(f"MAX1 기간: {MAX1_df.index.min()} ~ {MAX1_df.index.max()}")
    print(f"MAX5 기간: {MAX5_df.index.min()} ~ {MAX5_df.index.max()}")
else:
    # 파일이 없으면 잔차 계산 수행
    print(f"기존 파일이 없습니다. 잔차 계산을 시작합니다.")
    
    def get_residuals_monthly(y: pd.Series, X: pd.DataFrame) -> pd.Series:
        """
        각 월의 모든 데이터로 회귀를 수행하고, 
        그 회귀식을 바탕으로 해당 월의 모든 날짜에 대해 잔차를 계산
        
        y: 한 종목의 초과수익률 시계열
        X: FF3 팩터 DataFrame (MKT, SMB, HML)
        """
        # 인덱스 정렬 및 공통 인덱스만 사용
        common_idx = y.index.intersection(X.index)
        if len(common_idx) == 0:
            return pd.Series(index=y.index, dtype=float)
        
        y_aligned = y.loc[common_idx]
        X_aligned = X.loc[common_idx]
        
        # 데이터 병합
        df = pd.concat([y_aligned.rename("y"), X_aligned], axis=1)
        
        # 결과를 저장할 Series (공통 인덱스 기준)
        residual_series = pd.Series(index=df.index, dtype=float)
        
        # 월별로 그룹화하여 처리
        df['year_month'] = df.index.to_period('M')
        
        for year_month, group_df in df.groupby('year_month'):
            # 해당 월의 데이터에서 NaN 제거 (회귀에 사용할 데이터)
            month_data = group_df[['y', 'MKT', 'SMB', 'HML']].dropna()
            
            # 데이터가 충분하지 않으면 스킵 (최소 20개 이상 필요)
            if len(month_data) < 20:
                continue
            
            # 회귀 수행 (해당 월의 모든 데이터 사용, FF3 팩터만)
            X_month_df = month_data[['MKT','SMB','HML']].copy()
            X_month = sm.add_constant(X_month_df, has_constant='add')
            y_month = month_data['y'].values
            
            # X_month가 DataFrame인지 확인하고 numpy array로 변환
            if isinstance(X_month, pd.DataFrame):
                X_month_array = X_month.values
            else:
                X_month_array = X_month
            
            try:
                model = sm.OLS(y_month, X_month_array).fit()
                # 회귀 계수 추출 (intercept, MKT, SMB, HML)
                if hasattr(model.params, 'values'):
                    coef = model.params.values
                else:
                    coef = np.array(model.params)
            except:
                continue
            
            # 해당 월의 모든 날짜에 대해 잔차 계산 (벡터화 처리)
            # 회귀에 사용된 날짜와 사용되지 않은 날짜 모두 포함
            month_all_dates = group_df.index
            month_df = df.loc[month_all_dates, ['y', 'MKT', 'SMB', 'HML']].copy()
            
            # NaN이 없는 행만 계산
            valid_mask = ~month_df.isna().any(axis=1)
            valid_dates = month_df.index[valid_mask]
            
            if len(valid_dates) > 0:
                valid_data = month_df.loc[valid_dates]
                # 예측값 계산: intercept + coef[1]*MKT + coef[2]*SMB + coef[3]*HML
                mkt_vals = valid_data['MKT'].values if hasattr(valid_data['MKT'], 'values') else np.array(valid_data['MKT'])
                smb_vals = valid_data['SMB'].values if hasattr(valid_data['SMB'], 'values') else np.array(valid_data['SMB'])
                hml_vals = valid_data['HML'].values if hasattr(valid_data['HML'], 'values') else np.array(valid_data['HML'])
                y_vals = valid_data['y'].values if hasattr(valid_data['y'], 'values') else np.array(valid_data['y'])
                
                y_pred = (coef[0] + 
                         coef[1] * mkt_vals + 
                         coef[2] * smb_vals + 
                         coef[3] * hml_vals)
                # 잔차 계산
                residual_series.loc[valid_dates] = y_vals - y_pred
        
        return residual_series

    # 모든 종목(컬럼) 잔차 계산 → residual_df 생성
    tickers = excess_return.columns.tolist()
    print(f"총 {len(tickers)}개 종목의 잔차 계산 시작...")

    # 병렬 처리 (tqdm은 병렬 처리와 호환성 문제가 있어 제거)
    residual_list = Parallel(n_jobs=-1, verbose=10)(
        delayed(get_residuals_monthly)(excess_return[ticker], X)
        for ticker in tickers
    )

    residual_df = pd.concat(residual_list, axis=1)
    residual_df.columns = tickers

    print("잔차 계산 완료!")
    print("\n결과 예시:")
    print(residual_df.head())
    
    # ----- 월별 MAX1, MAX5 계산 함수 -----
    def calc_MAX_factors(residual_df):
        # 월별 MAX1: 각 월에서 각 종목별 최대값
        max1 = residual_df.resample('ME').max()
        # 월별 MAX5: 각 월에서 각 종목별 상위 5개 평균
        def max5_func(df):
            return df.apply(lambda s: s.sort_values(ascending=False).head(5).mean())
        max5 = residual_df.resample('ME').apply(max5_func)
        max1.columns = residual_df.columns
        max5.columns = residual_df.columns
        return max1, max5

    # ----- 실제 MAX1, MAX5 계산 -----
    MAX1_df, MAX5_df = calc_MAX_factors(residual_df)

    # 결과 확인
    print("\nMAX1 예시")
    print(MAX1_df.head(5))
    print("\nMAX5 예시")
    print(MAX5_df.head())

기존 파일이 없습니다. 잔차 계산을 시작합니다.
총 3892개 종목의 잔차 계산 시작...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    7.9s
[Parallel(n_jobs=-1)]: Done   6 tasks      | elapsed:    8.0s
[Parallel(n_jobs=-1)]: Done  13 tasks      | elapsed:    8.1s
[Parallel(n_jobs=-1)]: Done  20 tasks      | elapsed:    8.1s
[Parallel(n_jobs=-1)]: Done  29 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done  38 tasks      | elapsed:    8.3s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1808239251167252s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed:    8.4s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.09811520576477051s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done  67 tasks      | elapsed:    8.5s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.07191944122314453s.) Setting batch_size=8.
[Parallel(n_jobs=-1)]: Done 106 tasks      | elapsed:    8.7s
[Parallel(n_jobs=-1)]: Done 190 tasks      | elapsed:    9.0s
[Parallel(n_jobs=

잔차 계산 완료!

결과 예시:
                삼성전자    SK하이닉스  LG에너지솔루션  삼성바이오로직스       현대차   두산에너빌리티  \
Date                                                                     
2025-11-03 -0.017132  0.035313 -0.017066 -0.004740 -0.001017 -0.027737   
2025-11-04 -0.021557 -0.005556  0.027113  0.007446 -0.034255  0.019880   
2025-11-05  0.001297  0.044387  0.011160  0.008467 -0.008604 -0.020606   
2025-11-06 -0.015581  0.012624  0.003721  0.009008 -0.004332 -0.022230   
2025-11-07  0.009654  0.004442  0.002075  0.004147 -0.004807  0.022792   

                KB금융   HD현대중공업        기아  한화에어로스페이스  ...  성융광전투자  완리  골든센츄리  \
Date                                                 ...                      
2025-11-03  0.017488 -0.029388 -0.009820   0.030976  ...     NaN NaN    NaN   
2025-11-04  0.039749 -0.041674 -0.014755   0.002395  ...     NaN NaN    NaN   
2025-11-05 -0.000958 -0.035799 -0.022302  -0.019133  ...     NaN NaN    NaN   
2025-11-06  0.005979  0.006720 -0.009041   0.020626  ...     NaN NaN

##### **MAX 계산**

In [ ]:
import os

# ----- 월별 MAX1, MAX5 계산 함수 -----
def calc_MAX_factors(residual_df):
    # 월별 MAX1: 각 월에서 각 종목별 최대값
    max1 = residual_df.resample('ME').max()
    # 월별 MAX5: 각 월에서 각 종목별 상위 5개 평균
    def max5_func(df):
        return df.apply(lambda s: s.sort_values(ascending=False).head(5).mean())
    max5 = residual_df.resample('ME').apply(max5_func)
    max1.columns = residual_df.columns
    max5.columns = residual_df.columns
    return max1, max5

# output/FF3 폴더에 MAX1.csv, MAX5.csv 파일이 있는지 확인
output_dir = './output/FF3'
max1_file = os.path.join(output_dir, 'MAX1 (251130).csv')
max5_file = os.path.join(output_dir, 'MAX5 (251130).csv')

if os.path.exists(max1_file) and os.path.exists(max5_file):
    # 파일이 있으면 계산을 건너뛰고 파일을 읽어옴
    print(f"기존 MAX1, MAX5 파일이 존재합니다. 계산을 건너뜁니다.")
    print(f"MAX1.csv 로드 중: {max1_file}")
    MAX1_df = pd.read_csv(max1_file, index_col=0, parse_dates=True)
    print(f"MAX5.csv 로드 중: {max5_file}")
    MAX5_df = pd.read_csv(max5_file, index_col=0, parse_dates=True)
    print("기존 MAX1, MAX5 데이터 로드 완료!")
else:
    # 파일이 없으면 MAX1, MAX5 계산 수행
    print(f"기존 파일이 없습니다. MAX1, MAX5 계산을 시작합니다.")
    
    # ----- 실제 MAX1, MAX5 계산 -----
    MAX1_df, MAX5_df = calc_MAX_factors(residual_df)
    
    print("MAX1, MAX5 계산 완료!")

# 결과 확인
print("\nMAX1 예시")
print(MAX1_df.head(5))
print("\nMAX5 예시")
print(MAX5_df.head())

기존 파일이 없습니다. MAX1, MAX5 계산을 시작합니다.
MAX1, MAX5 계산 완료!

MAX1 예시
               삼성전자    SK하이닉스  LG에너지솔루션  삼성바이오로직스       현대차   두산에너빌리티  \
Date                                                                    
2025-11-30  0.02503  0.044387  0.036064  0.018275  0.019693  0.043757   

                KB금융  HD현대중공업        기아  한화에어로스페이스  ...  성융광전투자  완리  골든센츄리  \
Date                                                ...                      
2025-11-30  0.039749  0.06099  0.022288    0.04968  ...     NaN NaN    NaN   

            평산차업 KDR  네프로아이티  중국고섬  SBI모기지  SBI핀테크솔루션즈  SNK  테라뷰 홀딩스  
Date                                                                  
2025-11-30       NaN     NaN   NaN     NaN         NaN  NaN      NaN  

[1 rows x 3892 columns]

MAX5 예시
                삼성전자    SK하이닉스  LG에너지솔루션  삼성바이오로직스       현대차   두산에너빌리티  \
Date                                                                     
2025-11-30  0.017826  0.033106  0.021596  0.010856  0.014126  0.026088   

             

In [ ]:
##### **MAX1, MAX5 DataFrame 저장**

import os

# output 폴더 생성 (없으면 생성)
output_dir = './output/FF3'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"'{output_dir}' 폴더가 생성되었습니다.")

# 저장 파일 경로 설정
max1_file = os.path.join(output_dir, 'MAX1 (251130).csv')
max5_file = os.path.join(output_dir, 'MAX5 (251130).csv')

# CSV 파일로 저장
MAX1_df.to_csv(max1_file, encoding='utf-8-sig')
MAX5_df.to_csv(max5_file, encoding='utf-8-sig')

print("=" * 60)
print("MAX1, MAX5 DataFrame 저장 완료")
print("=" * 60)
print(f"MAX1 저장 경로: {max1_file}")
print(f"MAX1 shape: {MAX1_df.shape}")
print(f"MAX1 기간: {MAX1_df.index.min()} ~ {MAX1_df.index.max()}")
print()
print(f"MAX5 저장 경로: {max5_file}")
print(f"MAX5 shape: {MAX5_df.shape}")
print(f"MAX5 기간: {MAX5_df.index.min()} ~ {MAX5_df.index.max()}")
print()

MAX1, MAX5 DataFrame 저장 완료
MAX1 저장 경로: ./output/FF3\MAX1 (251101-251130).csv
MAX1 shape: (1, 3892)
MAX1 기간: 2025-11-30 00:00:00 ~ 2025-11-30 00:00:00

MAX5 저장 경로: ./output/FF3\MAX5 (251101-251130).csv
MAX5 shape: (1, 3892)
MAX5 기간: 2025-11-30 00:00:00 ~ 2025-11-30 00:00:00



---
## **0-2. 포트폴리오 계산**

##### **데이터 로드**

In [ ]:
import pandas as pd
import numpy as np

# 포트폴리오 수익률 계산을 위한 총수익률 데이터
total_adj_close  = pd.read_csv("./input/KOSPI_KOSDAQ_total_adj_close.csv", index_col=0, parse_dates=True)   # 현금배당 반영 수정주가(총수익률 계산)

# 시가총액가중 배분비중 계산 데이터
mkt_cap          = pd.read_csv("./input/KOSPI_KOSDAQ_mkt_cap.csv", index_col=0, parse_dates=True)           # 시가총액

# 거래대금 하위 10% 필터링 계산 데이터
trading_value_60 = pd.read_csv("./input/KOSPI_KOSDAQ_trading_value_60.csv", index_col=0, parse_dates=True)  # 60영업일 평균 거래대금

# Amihud illiquidity 계산 데이터
daily_ret        = pd.read_csv("./input/KOSPI_KOSDAQ_daily_ret.csv", index_col=0, parse_dates=True)         # 일간 수익률 월별 데이터
trading_value    = pd.read_csv("./input/KOSPI_KOSDAQ_trading_value.csv", index_col=0, parse_dates=True)     # 월말 거래대금

##### **함수 정의**

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm

# 함수 정의
def run_strategy(wins_threshold_temp, weight_method_temp, cost_temp,
                 trading_threshold_temp, Amihud_threshold_temp, n_temp):
    
    high_cost, low_cost = cost_temp
    NAV = initial_NAV

    portfolio_name   = f"{weight_method_temp}_R{wins_threshold_temp}_H{int(high_cost*10000)}L{int(low_cost*10000)}_T{trading_threshold_temp}_A{Amihud_threshold_temp}_Q({n_temp+1}/{q_cut})"
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade      = pd.Series(dtype=float, name=portfolio_name)

    # 수익률 윈저라이징
    ret_wins = ret_m.clip(
        lower = ret_m.quantile(wins_threshold_temp),
        upper = ret_m.quantile(1 - wins_threshold_temp),
        axis  = 1
    )

    prev_portfolio = None
    
    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        # 거래대금 하위 x% 필터링
        series          = trading_value_60.loc[start_date]
        threshold       = series.quantile(trading_threshold_temp)
        filtered        = series[series > threshold]
        factor_filtered = factor_df.loc[start_date, filtered.index]

        # 종목 선정 (분위수)
        quantiles = pd.qcut(factor_filtered, q=q_cut, labels=False)
        basket    = factor_filtered[quantiles == n_temp]

        if basket.empty:
            continue

        # Amihud illiquidity 계산
        illiq        = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
        threshold    = illiq.quantile(1 - Amihud_threshold_temp)
        illiquid_top = illiq[illiq >= threshold].index

        # 이전 비중
        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket.index)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        # 목표 비중
        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1/len(basket), index=basket.index)
        else:  # Cap
            cap_seg = mkt_cap.loc[start_date, basket.index]
            target_weights = cap_seg / cap_seg.sum()

        # 거래비용 반영
        all_index     = target_weights.index.union(prev_weights.index)
        target_w      = target_weights.reindex(all_index, fill_value=0)
        prev_w        = prev_weights.reindex(all_index, fill_value=0)
        delta_w       = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate     = np.where(delta_w.index.isin(illiquid_top), high_cost, low_cost)
        trade_cost    = (trade_amounts * cost_rate).sum()
        NAV_new       = NAV - trade_cost

        # 포트폴리오 가치
        current_portfolio_value = target_weights * NAV_new
        ret_seg                 = ret_wins.loc[end_date, basket.index]
        next_portfolio_value    = current_portfolio_value * (ret_seg + 1)

        NAV_new       = next_portfolio_value.sum()
        portfolio_ret = NAV_new / NAV - 1

        prev_portfolio = next_portfolio_value
        NAV            = NAV_new

        total_trade.loc[start_date]    = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    return portfolio_return, total_trade

# 포트폴리오 선택 함수
def select_columns(df, fixed_options):
    selected = []
    new_names = {}
    
    for c in df.columns:
        parts = c.split('_')
        
        # 조건 체크 (고정한 옵션은 정확히 일치해야 함)
        if all(parts[i] == v or v == "*" for i, v in fixed_options.items()):
            selected.append(c)
            
            # 새 이름은 "고정하지 않은 옵션만"
            free_parts = [parts[i] for i, v in fixed_options.items() if v == "*"]
            new_name = "_".join(free_parts)
            new_names[c] = new_name
    
    # 선택된 subset 반환, 컬럼명은 자유 요소만
    subset = df[selected].rename(columns=new_names)
    return subset

---
##### **설정**

In [145]:
# 팩터 설정
factor_df         = MAX5_df

In [146]:
start_point       = '2000-01-01'                     # 백테스트 기간 설정
end_point         = '2025-10-31'

q_cut             = 10                               # 포트폴리오를 나누는 분위 수 e.g. 5 → 5분위 
n                 = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]                  # 0 : 하위 ~ (q_cut - 1) : 상위    e.g. [0, 1, 2, 3, 4]

weight_method     = ['Equal', 'Cap']                 # ['Equal', Cap']

cost              = [(0.008, 0.003), (0, 0)]         # (high_cost, low_cost)            e.g. [(0.008, 0.003), (0, 0)]
                                                     
wins_threshold    = [0.01]                           # 수익률 윈저라이징                  e.g. [0.01, 0.05]
trading_threshold = [0.1]                            # 60일 평균 거래대금 필터링           e.g. [0.1, 0.0]
Amihud_threshold  = [0.2]                            # 유동성 기준                       e.g. [0.2, 0.0]

initial_NAV       = 1                                # 초기값

##### **계산 실행**

In [147]:
# ret_m        = total_adj_close.resample('QE').last().ffill().pct_change() # 분기 리밸런싱
ret_m        = total_adj_close.resample('ME').last().ffill().pct_change()

portfolio_df = pd.DataFrame()
trade_df     = pd.DataFrame()

month_ends   = factor_df.resample('ME').last().index
month_ends   = month_ends[(month_ends >= pd.to_datetime(start_point)) &
                        (month_ends <= pd.to_datetime(end_point))]

C:\Users\XH58\AppData\Local\Temp\ipykernel_11812\417012265.py:2: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [148]:
# 모든 조합 만들기
param_list = [
    (wins_threshold_temp, weight_method_temp, cost_temp, trading_threshold_temp, Amihud_threshold_temp, n_temp)
    for wins_threshold_temp in wins_threshold
    for weight_method_temp in weight_method
    for cost_temp in cost
    for trading_threshold_temp in trading_threshold
    for Amihud_threshold_temp in Amihud_threshold
    for n_temp in n
]

# 병렬 실행
results = Parallel(n_jobs=-1)(
    delayed(run_strategy)(*params) for params in tqdm(param_list)
)

# 결과 합치기
portfolio_df = pd.DataFrame({r[0].name: r[0] for r in results})
trade_df     = pd.DataFrame({r[1].name: r[1] for r in results})

100%|██████████| 40/40 [00:56<00:00,  1.41s/it]


##### **Output**

In [149]:
portfolio_df.tail()

,Equal_R0.01_H80L30_T0.1_A0.2_Q(1/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(2/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(3/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(4/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(5/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(6/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(7/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(8/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(9/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(10/10),...,Cap_R0.01_H0L0_T0.1_A0.2_Q(1/10),Cap_R0.01_H0L0_T0.1_A0.2_Q(2/10),Cap_R0.01_H0L0_T0.1_A0.2_Q(3/10),Cap_R0.01_H0L0_T0.1_A0.2_Q(4/10),Cap_R0.01_H0L0_T0.1_A0.2_Q(5/10),Cap_R0.01_H0L0_T0.1_A0.2_Q(6/10),Cap_R0.01_H0L0_T0.1_A0.2_Q(7/10),Cap_R0.01_H0L0_T0.1_A0.2_Q(8/10),Cap_R0.01_H0L0_T0.1_A0.2_Q(9/10),Cap_R0.01_H0L0_T0.1_A0.2_Q(10/10)
2025-06-30,0.038437,0.031914,0.051287,0.046059,0.037627,0.057155,0.059021,0.077387,0.068886,0.010874,...,0.124230,0.061691,0.148243,0.069237,0.238934,0.133464,0.083869,0.096869,0.214863,0.045110
2025-07-31,0.011956,0.014864,0.027501,0.015211,0.020217,0.009607,0.003865,0.012420,-0.010204,-0.046856,...,0.093273,0.028739,0.060715,0.097104,0.026727,0.030247,0.041249,0.065188,0.017882,-0.044731
2025-08-31,-0.022279,-0.019861,-0.025878,-0.030608,-0.012500,-0.030009,-0.037449,-0.010227,-0.024206,-0.030376,...,-0.021214,-0.025037,0.008119,-0.032678,-0.021414,-0.010181,-0.018509,-0.014046,-0.018157,0.000871
2025-09-30,-0.002159,-0.005102,0.009997,0.019586,0.021581,0.030008,0.037730,0.033226,0.036203,0.019790,...,0.011052,0.015037,0.155124,0.051425,0.083327,0.006419,0.177140,0.023846,0.022146,0.028833
2025-10-31,-0.020050,-0.001919,-0.015734,0.013585,0.004829,-0.004281,-0.004447,0.028469,-0.002442,0.033844,...,-0.015735,0.132687,0.204017,0.351403,0.173008,0.095380,0.084425,0.177865,0.057843,0.085411


---
## **1. 분위수 검증**

##### **포트폴리오 선택**

In [150]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

subset = select_columns(
    portfolio_df,
    {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

In [151]:
subset.tail()

,Q(1/10),Q(2/10),Q(3/10),Q(4/10),Q(5/10),Q(6/10),Q(7/10),Q(8/10),Q(9/10),Q(10/10)
2025-06-30,0.038437,0.031914,0.051287,0.046059,0.037627,0.057155,0.059021,0.077387,0.068886,0.010874
2025-07-31,0.011956,0.014864,0.027501,0.015211,0.020217,0.009607,0.003865,0.012420,-0.010204,-0.046856
2025-08-31,-0.022279,-0.019861,-0.025878,-0.030608,-0.012500,-0.030009,-0.037449,-0.010227,-0.024206,-0.030376
2025-09-30,-0.002159,-0.005102,0.009997,0.019586,0.021581,0.030008,0.037730,0.033226,0.036203,0.019790
2025-10-31,-0.020050,-0.001919,-0.015734,0.013585,0.004829,-0.004281,-0.004447,0.028469,-0.002442,0.033844


##### **Code**

In [152]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 데이터 준비
nav = (subset + 1).cumprod()
log_cum = np.log1p(subset).cumsum()

# 서브플롯 (가로 2개)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Cumulative NAV", "Log Cumulative Return"])

# 첫 번째 그래프
for col in nav.columns:
    fig.add_trace(
        go.Scatter(x=nav.index, y=nav[col], mode='lines', name=col,
                   legendgroup=col, showlegend=True),
        row=1, col=1
    )

# 두 번째 그래프
for col in log_cum.columns:
    fig.add_trace(
        go.Scatter(x=log_cum.index, y=log_cum[col], mode='lines', name=col,
                   legendgroup=col, showlegend=True),
        row=1, col=2
    )

# 레이아웃 기본 설정
fig.update_layout(
    title="Portfolio Performance Comparison",
    height=500, width=1000,
    template="plotly_white"
)

fig.update_layout(
    shapes=[
        # 왼쪽 그래프 영역 (x=0~0.45, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.0, y0=0.0, x1=0.45, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
        
        # 오른쪽 그래프 영역 (x=0.55~1, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.55, y0=0.0, x1=1.0, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
    ]
)

##### **Plot**

In [42]:
fig.show()

---
## **2. Double sort : Size Control**

##### **함수**

In [ ]:
def run_strategy(c_temp, n_temp,
                 wins_threshold_temp, weight_method_temp, cost_temp,
                 trading_threshold_temp, Amihud_threshold_temp, 
                 initial_NAV=1):
    high_cost, low_cost = cost_temp
    NAV = initial_NAV
    portfolio_name = f"C({c_temp+1}/{cap_cut})_Q({n_temp+1}/{q_cut})"
    
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade      = pd.Series(dtype=float, name=portfolio_name)
    
    # 수익률 윈저라이징
    ret_wins = ret_m.clip(
        lower = ret_m.quantile(wins_threshold_temp),
        upper = ret_m.quantile(1 - wins_threshold_temp),
        axis  = 1
    )
    
    prev_portfolio = None

    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        cap_series    = mkt_cap.loc[start_date]
        cap_quantiles = pd.qcut(cap_series, q=cap_cut, labels=False)
        cap_filtered  = cap_series[cap_quantiles == c_temp]

        series    = trading_value_60.loc[start_date, cap_filtered.index]
        threshold = series.quantile(trading_threshold_temp)
        filtered  = series[series > threshold]
        factor_filtered = factor_df.loc[start_date, filtered.index]

        quantiles = pd.qcut(factor_filtered, q=q_cut, labels=False)
        basket    = factor_filtered[quantiles == n_temp]
        if basket.empty:
            continue

        illiq        = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
        threshold    = illiq.quantile(1 - Amihud_threshold_temp)
        illiquid_top = illiq[illiq >= threshold].index

        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket.index)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1/len(basket), index=basket.index)
        else:  # Cap
            cap_seg = mkt_cap.loc[start_date, basket.index]
            target_weights = cap_seg / cap_seg.sum()

        all_index     = target_weights.index.union(prev_weights.index)
        target_w      = target_weights.reindex(all_index, fill_value=0)
        prev_w        = prev_weights.reindex(all_index, fill_value=0)
        delta_w       = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate     = np.where(delta_w.index.isin(illiquid_top), high_cost, low_cost)
        trade_cost    = (trade_amounts * cost_rate).sum()
        NAV_new       = NAV - trade_cost

        current_portfolio_value = target_weights * NAV_new
        ret_seg = ret_wins.loc[end_date, basket.index]
        next_portfolio_value = current_portfolio_value * (ret_seg + 1)

        NAV_new       = next_portfolio_value.sum()
        portfolio_ret = NAV_new / NAV - 1

        prev_portfolio = next_portfolio_value
        NAV            = NAV_new

        total_trade.loc[start_date]    = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    return portfolio_return, total_trade


##### **계산 실행**

In [44]:
cap_cut           = 5                                # 포트폴리오를 나누는 분위 수 e.g. 5 → 5분위 
c                 = [0, 1, 2, 3, 4]    

In [45]:
trading_threshold_temp = trading_threshold[0]
weight_method_temp     = weight_method[0]
cost_temp              = cost[0]
wins_threshold_temp    = wins_threshold[0]
Amihud_threshold_temp  = Amihud_threshold[0]

In [46]:
# --- 병렬 실행 ---
results = Parallel(n_jobs=-1)(
    delayed(run_strategy)(
        c_temp, n_temp,
        wins_threshold_temp, weight_method_temp, cost_temp,
        trading_threshold_temp, Amihud_threshold_temp,
        initial_NAV
    )
    for c_temp in c
    for n_temp in n
)

# --- 결과 합치기 ---
cap_cut_profolio_df = pd.DataFrame()
for (portfolio_return, total_trade) in results:
    cap_cut_profolio_df[portfolio_return.name] = portfolio_return

##### **Output**

In [47]:
cap_cut_profolio_df.tail()

,C(1/5)_Q(1/5),C(1/5)_Q(2/5),C(1/5)_Q(3/5),C(1/5)_Q(4/5),C(1/5)_Q(5/5),C(2/5)_Q(1/5),C(2/5)_Q(2/5),C(2/5)_Q(3/5),C(2/5)_Q(4/5),C(2/5)_Q(5/5),...,C(4/5)_Q(1/5),C(4/5)_Q(2/5),C(4/5)_Q(3/5),C(4/5)_Q(4/5),C(4/5)_Q(5/5),C(5/5)_Q(1/5),C(5/5)_Q(2/5),C(5/5)_Q(3/5),C(5/5)_Q(4/5),C(5/5)_Q(5/5)
2025-06-30,0.002447,0.034999,0.023935,0.034468,0.002460,0.025527,0.059106,0.043254,0.057550,0.034023,...,0.047834,0.041927,0.063658,0.057959,0.049593,0.084507,0.084007,0.108036,0.103309,0.112094
2025-07-31,0.000607,0.004514,0.014347,-0.004136,-0.024934,0.011798,0.019706,0.003375,0.003120,-0.023625,...,0.029632,0.029594,0.029101,0.016561,-0.034055,0.047525,0.040766,0.023219,0.031360,-0.029999
2025-08-31,-0.009839,-0.006817,-0.003160,-0.015289,-0.033250,-0.013182,-0.021688,-0.022072,-0.036602,-0.037022,...,-0.028164,-0.023796,-0.025919,-0.028710,-0.022402,-0.031248,-0.024146,-0.027978,-0.003274,0.009761
2025-09-30,0.004201,0.001092,-0.008942,0.012045,0.023617,-0.008477,-0.013097,0.009925,0.026439,-0.006190,...,-0.000337,0.056184,0.050501,0.041265,0.053175,0.015527,0.050995,0.028136,0.061308,0.070855
2025-10-31,-0.003169,-0.026459,-0.032969,-0.025772,-0.032432,-0.026778,-0.020196,-0.039417,-0.035147,-0.044934,...,0.005114,-0.000931,0.000968,0.012796,0.049214,0.041499,0.110523,0.047258,0.103723,0.071779


---
##### **Plot**

##### **Code**

In [48]:
def CAGR(series, periods_per_year=12):
    """월별 수익률 시리즈 기준 CAGR 계산"""
    series = series.dropna()
    if series.empty:
        return np.nan
    cumulative = (1 + series).cumprod()
    start, end = cumulative.index[0], cumulative.index[-1]
    n_years = (end - start).days / 365.25
    return (cumulative.iloc[-1] ** (1 / n_years)) - 1

In [49]:
# heatmap용 DF 초기화
cap_cut = 5
q_cut   = 5
cagr_matrix = pd.DataFrame(index=[f"C({i+1}/{cap_cut})" for i in range(cap_cut)],
                           columns=[f"Q({j+1}/{q_cut})" for j in range(q_cut)])

for col in cap_cut_profolio_df.columns:
    # 이름 parsing: "C(i/5)_Q(j/5)"
    c_part, q_part = col.split('_')
    cagr_matrix.loc[c_part, q_part] = CAGR(cap_cut_profolio_df[col])
    
cagr_matrix = cagr_matrix.astype(float)

In [50]:
import plotly.express as px

fig = px.imshow(
    cagr_matrix.astype(float),
    text_auto=".1%",
    color_continuous_scale="Blues",  # 파란 계열
    aspect="equal",                  # 정사각형 셀
    labels=dict(
        x="Factor Quintile (Q)", 
        y="Market Cap Quintile (C)", 
        color="CAGR"
    )
)

fig.update_layout(
    title="CAGR Heatmap by Cap vs Factor Quintile",
    xaxis=dict(side="top"),
    width=600,   # 가로 크기
    height=600,  # 세로 크기
    margin=dict(l=60, r=60, t=80, b=60)  # 여백 조정
)

fig.show()

##### **Heatmap**

In [51]:
fig.show()

In [52]:
factors        = pd.read_csv("./input/factors_monthly.csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)
portfolio      = cap_cut_profolio_df['C(1/5)_Q(1/5)']
portfolio.name = 'Return'

In [53]:
# 1. 두 데이터 공통 구간 맞추기
df = pd.concat([portfolio, factors], axis=1, join="inner").dropna()

# 2. 종속변수: 포트폴리오 초과수익률
y = df['Return'] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM']
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀 (Newey-West 표준오차)
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})

# 5. 결과 출력
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.737
Model:                            OLS   Adj. R-squared:                  0.734
Method:                 Least Squares   F-statistic:                     78.53
Date:                Fri, 14 Nov 2025   Prob (F-statistic):           1.11e-45
Time:                        13:35:14   Log-Likelihood:                 605.99
No. Observations:                 309   AIC:                            -1202.
Df Residuals:                     304   BIC:                            -1183.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0128      0.003      4.226      0.0

In [54]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 포트폴리오 설정
portfolio = cap_cut_profolio_df['C(1/5)_Q(1/5)']

# 데이터 준비
nav = (1 + portfolio).cumprod()  # 누적 NAV
log_cum = np.log1p(portfolio).cumsum()  # 로그 누적 수익률

# 서브플롯 (가로 2개)
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=["Cumulative NAV", "Log Cumulative Return"],
    horizontal_spacing=0.15
)

# 첫 번째 그래프: 누적 NAV
fig.add_trace(
    go.Scatter(
        x=nav.index, 
        y=nav.values, 
        mode='lines', 
        name='C(1/5)_Q(1/5)',
        line=dict(color='#1f77b4', width=2)
    ),
    row=1, col=1
)

# 두 번째 그래프: 로그 누적 수익률
fig.add_trace(
    go.Scatter(
        x=log_cum.index, 
        y=log_cum.values, 
        mode='lines', 
        name='C(1/5)_Q(1/5)',
        line=dict(color='#1f77b4', width=2),
        showlegend=False
    ),
    row=1, col=2
)

# 레이아웃 설정
fig.update_layout(
    title="Portfolio Performance: C(1/5)_Q(1/5)",
    height=500, 
    width=1000,
    template="plotly_white",
    hovermode='x unified'
)

# 축 레이블 설정
fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_xaxes(title_text="Date", row=1, col=2)
fig.update_yaxes(title_text="Cumulative NAV", row=1, col=1)
fig.update_yaxes(title_text="Log Cumulative Return", row=1, col=2)

fig.show()

In [55]:
portfolio.to_csv('./output/FF3/portfolio_C(1-5)_Q(1-5).csv', index=True)

In [56]:
weight_method     = ['Equal', 'Cap']                 # ['Equal', Cap']

cost              = [(0.008, 0.003), (0, 0)]         # (high_cost, low_cost)            e.g. [(0.008, 0.003), (0, 0)]
                                                     
wins_threshold    = [0.01]                           # 수익률 윈저라이징                  e.g. [0.01, 0.05]
trading_threshold = [0.1]                            # 60일 평균 거래대금 필터링           e.g. [0.1, 0.0]
Amihud_threshold  = [0.2]  

In [57]:
def run_portfolio_cap1_max1(wins_threshold_temp, weight_method_temp, cost_temp,
                            trading_threshold_temp, Amihud_threshold_temp,
                            include_cost=True, initial_NAV=1):
    """
    Cap(1/5)와 MAX(1/5) 포트폴리오 구성 함수
    
    Parameters
    ----------
    wins_threshold_temp : float
        수익률 윈저라이징 임계값
    weight_method_temp : str
        'Equal' 또는 'Cap'
    cost_temp : tuple
        (high_cost, low_cost) 거래비용
    trading_threshold_temp : float
        거래대금 필터링 임계값
    Amihud_threshold_temp : float
        Amihud illiquidity 임계값
    include_cost : bool
        True면 거래비용 반영, False면 미반영
    initial_NAV : float
        초기 NAV
    
    Returns
    -------
    tuple
        (portfolio_return, total_trade)
    """
    high_cost, low_cost = cost_temp
    NAV = initial_NAV
    
    # 포트폴리오 이름 설정
    cost_suffix = "WithCost" if include_cost else "NoCost"
    portfolio_name = f"Cap1MAX1_{weight_method_temp}_R{wins_threshold_temp}_H{int(high_cost*10000)}L{int(low_cost*10000)}_T{trading_threshold_temp}_A{Amihud_threshold_temp}_{cost_suffix}"
    
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade = pd.Series(dtype=float, name=portfolio_name)
    
    # 수익률 윈저라이징
    ret_wins = ret_m.clip(
        lower=ret_m.quantile(wins_threshold_temp),
        upper=ret_m.quantile(1 - wins_threshold_temp),
        axis=1
    )
    
    prev_portfolio = None
    
    # Cap(1/5), MAX(1/5) 설정
    cap_cut = 5
    q_cut = 5
    c_temp = 0  # Cap 하위 20% (1/5)
    n_temp = 0  # MAX 하위 20% (1/5)
    
    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]
        
        # Cap(1/5) 필터링: 시가총액 하위 20%
        cap_series = mkt_cap.loc[start_date]
        cap_quantiles = pd.qcut(cap_series, q=cap_cut, labels=False, duplicates='drop')
        cap_filtered = cap_series[cap_quantiles == c_temp]
        
        # 거래대금 필터링
        series = trading_value_60.loc[start_date, cap_filtered.index]
        threshold = series.quantile(trading_threshold_temp)
        filtered = series[series > threshold]
        factor_filtered = factor_df.loc[start_date, filtered.index]
        
        # MAX(1/5) 필터링: MAX 팩터 하위 20%
        quantiles = pd.qcut(factor_filtered, q=q_cut, labels=False, duplicates='drop')
        basket = factor_filtered[quantiles == n_temp]
        
        if basket.empty:
            continue
        
        # Amihud illiquidity 계산
        illiq = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
        threshold = illiq.quantile(1 - Amihud_threshold_temp)
        illiquid_top = illiq[illiq >= threshold].index
        
        # 이전 포트폴리오 가중치 계산
        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket.index)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()
        
        # 목표 가중치 계산
        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1/len(basket), index=basket.index)
        else:  # Cap
            cap_seg = mkt_cap.loc[start_date, basket.index]
            target_weights = cap_seg / cap_seg.sum()
        
        # 가중치 변화량 계산
        all_index = target_weights.index.union(prev_weights.index)
        target_w = target_weights.reindex(all_index, fill_value=0)
        prev_w = prev_weights.reindex(all_index, fill_value=0)
        delta_w = target_w - prev_w
        
        # 거래비용 계산
        trade_amounts = abs(delta_w) * NAV
        cost_rate = np.where(delta_w.index.isin(illiquid_top), high_cost, low_cost)
        
        if include_cost:
            trade_cost = (trade_amounts * cost_rate).sum()
        else:
            trade_cost = 0  # 거래비용 미반영
        
        # 포트폴리오 가치 계산
        NAV_new = NAV - trade_cost
        current_portfolio_value = target_weights * NAV_new
        ret_seg = ret_wins.loc[end_date, basket.index]
        next_portfolio_value = current_portfolio_value * (ret_seg + 1)
        
        # NAV 업데이트
        NAV_new = next_portfolio_value.sum()
        portfolio_ret = NAV_new / NAV - 1
        
        prev_portfolio = next_portfolio_value
        NAV = NAV_new
        
        total_trade.loc[start_date] = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret
    
    return portfolio_return, total_trade


# 포트폴리오 구성 실행
# 설정값 (기존 코드에서 사용하는 값들을 참고)
trading_threshold_temp = trading_threshold[0]
weight_method_temp     = weight_method[0]
cost_temp              = cost[0]
wins_threshold_temp    = wins_threshold[0]
Amihud_threshold_temp  = Amihud_threshold[0]

# 거래비용 반영 포트폴리오
portfolio_return_with_cost, total_trade_with_cost = run_portfolio_cap1_max1(
    wins_threshold_temp=wins_threshold_temp,
    weight_method_temp=weight_method_temp,
    cost_temp=cost_temp,
    trading_threshold_temp=trading_threshold_temp,
    Amihud_threshold_temp=Amihud_threshold_temp,
    include_cost=True
)

# 거래비용 미반영 포트폴리오
portfolio_return_no_cost, total_trade_no_cost = run_portfolio_cap1_max1(
    wins_threshold_temp=wins_threshold_temp,
    weight_method_temp=weight_method_temp,
    cost_temp=cost_temp,
    trading_threshold_temp=trading_threshold_temp,
    Amihud_threshold_temp=Amihud_threshold_temp,
    include_cost=False
)

# 결과를 DataFrame으로 합치기
MAX1_CAP1_portfolio_df = pd.DataFrame({
    'WithCost': portfolio_return_with_cost,
    'NoCost': portfolio_return_no_cost
})

In [58]:
MAX1_CAP1_portfolio_df.tail()

,WithCost,NoCost
2025-06-30,0.002447,0.005736
2025-07-31,0.000607,0.003803
2025-08-31,-0.009839,-0.005667
2025-09-30,0.004201,0.008178
2025-10-31,-0.003169,0.000900


In [59]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

subset = select_columns(
    MAX1_CAP1_portfolio_df,
    {0: "*"}
)

In [60]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 데이터 준비
nav = (subset + 1).cumprod()
log_cum = np.log1p(subset).cumsum()

# 서브플롯 (가로 2개)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Cumulative NAV", "Log Cumulative Return"])

# 첫 번째 그래프
for col in nav.columns:
    fig.add_trace(
        go.Scatter(x=nav.index, y=nav[col], mode='lines', name=col,
                   legendgroup=col, showlegend=True),
        row=1, col=1
    )

# 두 번째 그래프
for col in log_cum.columns:
    fig.add_trace(
        go.Scatter(x=log_cum.index, y=log_cum[col], mode='lines', name=col,
                   legendgroup=col, showlegend=True),
        row=1, col=2
    )

# 레이아웃 기본 설정
fig.update_layout(
    title="Portfolio Performance Comparison",
    height=500, width=1000,
    template="plotly_white"
)

fig.update_layout(
    shapes=[
        # 왼쪽 그래프 영역 (x=0~0.45, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.0, y0=0.0, x1=0.45, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
        
        # 오른쪽 그래프 영역 (x=0.55~1, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.55, y0=0.0, x1=1.0, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
    ]
)

In [61]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

# 동일가중 포트폴리오
subset_cost = select_columns(
    MAX1_CAP1_portfolio_df,
    {0: "*"}
)

In [62]:
import numpy as np
import plotly.graph_objects as go

# 로그누적수익률 계산
log_cum_cost = np.log1p(subset_cost).cumsum()

# 단일 plot 생성
fig = go.Figure()

for col in log_cum_cost.columns:
    fig.add_trace(
        go.Scatter(
            x=log_cum_cost.index, 
            y=log_cum_cost[col], 
            mode='lines', 
            name=col
        )
    )

# 레이아웃 설정
fig.update_layout(
    title="Equal-weighted Portfolios (Cost Parameter Variation, Log Cumulative Return)",
    xaxis_title="Date",
    yaxis_title="Log Cumulative Return",
    template="plotly_white",
    height=600,
    width=900
)

In [63]:
import numpy as np

# 포트폴리오 수익률 추출
portfolio_returns = MAX1_CAP1_portfolio_df['WithCost'].dropna()

if len(portfolio_returns) > 0:
    # 월별 수익률 기준 계산
    monthly_returns = portfolio_returns
    
    # 총 기간 (년)
    total_months = len(monthly_returns)
    total_years = total_months / 12
    
    # NAV 계산
    nav = (1 + monthly_returns).cumprod()
    
    # CAGR (연평균 수익률)
    total_return = nav.iloc[-1] / nav.iloc[0] - 1
    cagr = (1 + total_return) ** (1 / total_years) - 1 if total_years > 0 else np.nan
    
    # 연화 표준편차 (월별 표준편차 * sqrt(12))
    monthly_std = monthly_returns.std()
    annual_volatility = monthly_std * np.sqrt(12)
    
    # 샤프 비율 (무위험 수익률 0 가정)
    sharpe_ratio = cagr / annual_volatility if annual_volatility != 0 else np.nan
    
    print(f"포트폴리오: MAX1_CAP1_R0.01_H80L30_T0.1_A0.2")
    print(f"CAGR (연평균 수익률): {cagr:.4f} ({cagr*100:.2f}%)")
    print(f"연화 표준편차: {annual_volatility:.4f} ({annual_volatility*100:.2f}%)")
    print(f"샤프 비율: {sharpe_ratio:.4f}")
else:
    print("데이터가 없습니다.")

print()

# 포트폴리오 수익률 추출
portfolio_returns = MAX1_CAP1_portfolio_df['NoCost'].dropna()

if len(portfolio_returns) > 0:
    # 월별 수익률 기준 계산
    monthly_returns = portfolio_returns
    
    # 총 기간 (년)
    total_months = len(monthly_returns)
    total_years = total_months / 12
    
    # NAV 계산
    nav = (1 + monthly_returns).cumprod()
    
    # CAGR (연평균 수익률)
    total_return = nav.iloc[-1] / nav.iloc[0] - 1
    cagr = (1 + total_return) ** (1 / total_years) - 1 if total_years > 0 else np.nan
    
    # 연화 표준편차 (월별 표준편차 * sqrt(12))
    monthly_std = monthly_returns.std()
    annual_volatility = monthly_std * np.sqrt(12)
    
    # 샤프 비율 (무위험 수익률 0 가정)
    sharpe_ratio = cagr / annual_volatility if annual_volatility != 0 else np.nan
    
    print(f"포트폴리오: MAX1_CAP1_R0.01_H0L0_T0.1_A0.2")
    print(f"CAGR (연평균 수익률): {cagr:.4f} ({cagr*100:.2f}%)")
    print(f"연화 표준편차: {annual_volatility:.4f} ({annual_volatility*100:.2f}%)")
    print(f"샤프 비율: {sharpe_ratio:.4f}")
else:
    print("데이터가 없습니다.")

포트폴리오: MAX1_CAP1_R0.01_H80L30_T0.1_A0.2
CAGR (연평균 수익률): 0.2136 (21.36%)
연화 표준편차: 0.2307 (23.07%)
샤프 비율: 0.9257

포트폴리오: MAX1_CAP1_R0.01_H0L0_T0.1_A0.2
CAGR (연평균 수익률): 0.3052 (30.52%)
연화 표준편차: 0.2330 (23.30%)
샤프 비율: 1.3097


In [64]:
# MAX1_CAP1_portfolio_df를 CSV 파일로 저장
MAX1_CAP1_portfolio_df.to_csv('./output/FF3/MAX1_CAP1_portfolio.csv', index=True, encoding='utf-8-sig')
print("CSV 파일 저장 완료: ./output/FF3/MAX1_CAP1_portfolio.csv")

CSV 파일 저장 완료: ./output/FF3/MAX1_CAP1_portfolio.csv


---
## **3. Parameter Sensitivity**
`0-2. 포트폴리오 계산`에서 계산한 포트폴리오 `portfolio_df` 데이터 사용

---
#### **1) Weight sensitivity**

##### **포트폴리오 선택**

In [65]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

# 동일가중 포트폴리오
subset_equal = select_columns(
    portfolio_df,
    {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

# 시총가중 포트폴리오
subset_cap = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

##### **Code**

In [66]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 로그누적수익률 계산
log_cum_equal = np.log1p(subset_equal).cumsum()
log_cum_cap   = np.log1p(subset_cap).cumsum()

# 서브플롯 (가로 2개)
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=["Equal-weighted Portfolios", 
                    "Cap-weighted Portfolios"]
)

# Equal 그래프
for col in log_cum_equal.columns:
    fig.add_trace(
        go.Scatter(x=log_cum_equal.index, y=log_cum_equal[col], 
                   mode='lines', name=f"Equal_{col}",
                   legendgroup=col, showlegend=True),
        row=1, col=1
    )

# Cap 그래프
for col in log_cum_cap.columns:
    fig.add_trace(
        go.Scatter(x=log_cum_cap.index, y=log_cum_cap[col], 
                   mode='lines', name=f"Cap_{col}",
                   legendgroup=col, showlegend=True),
        row=1, col=2
    )

# 레이아웃 기본 설정
fig.update_layout(
    title="Equal vs Cap Weighted Portfolios (Log Cumulative Returns)",
    height=500, width=1000,
    template="plotly_white"
)

# 각 subplot 테두리 표시
fig.update_layout(
    shapes=[
        dict(type="rect", xref="paper", yref="paper",
             x0=0.0, y0=0.0, x1=0.45, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
        dict(type="rect", xref="paper", yref="paper",
             x0=0.55, y0=0.0, x1=1.0, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
    ]
)

##### **Plot**

In [67]:
fig.show()

---
#### **2) Cost sensitivity**

##### **포트폴리오 선택**

In [68]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

# 동일가중 포트폴리오
subset_cost = select_columns(
    portfolio_df,
    {0: "Equal", 1: "R0.01", 2: "*", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
)

##### **Code**

In [69]:
import numpy as np
import plotly.graph_objects as go

# 로그누적수익률 계산
log_cum_cost = np.log1p(subset_cost).cumsum()

# 단일 plot 생성
fig = go.Figure()

for col in log_cum_cost.columns:
    fig.add_trace(
        go.Scatter(
            x=log_cum_cost.index, 
            y=log_cum_cost[col], 
            mode='lines', 
            name=col
        )
    )

# 레이아웃 설정
fig.update_layout(
    title="Equal-weighted Portfolios (Cost Parameter Variation, Log Cumulative Return)",
    xaxis_title="Date",
    yaxis_title="Log Cumulative Return",
    template="plotly_white",
    height=600,
    width=900
)

##### **Plot**

In [70]:
fig.show()

---
## **4. 스프레드 회귀 (Factor regression)**

##### **데이터 로드**

In [71]:
factors        = pd.read_csv("./input/factors_monthly.csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)
portfolio      = portfolio_df['Equal_R0.01_H80L30_T0.1_A0.2_Q(1/5)']
portfolio.name = 'Return'

##### **함수 정의**

In [72]:
def run_strategy(c_temp, th,
                 wins_threshold_temp, weight_method_temp, cost_temp,
                 trading_threshold_temp, Amihud_threshold_temp, 
                 initial_NAV=1):
    high_cost, low_cost = cost_temp
    NAV = initial_NAV
    portfolio_name = f"C({c_temp+1}/{cap_cut})_Q({th})"
    
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade      = pd.Series(dtype=float, name=portfolio_name)
    
    # 수익률 윈저라이징
    ret_wins = ret_m.clip(
        lower = ret_m.quantile(wins_threshold_temp),
        upper = ret_m.quantile(1 - wins_threshold_temp),
        axis  = 1
    )
    
    prev_portfolio = None

    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        cap_series    = mkt_cap.loc[start_date]
        cap_quantiles = pd.qcut(cap_series, q=cap_cut, labels=False)
        cap_filtered  = cap_series[cap_quantiles == c_temp]

        series    = trading_value_60.loc[start_date, cap_filtered.index]
        threshold = series.quantile(trading_threshold_temp)
        filtered  = series[series > threshold]
        factor_filtered = factor_df.loc[start_date, filtered.index]

        low_th  = factor_filtered.quantile(0.3)
        high_th = factor_filtered.quantile(0.7)
        
        if th == 'long':   # IVOL 낮은 그룹
            basket = factor_filtered[factor_filtered <= low_th]
        else:              # IVOL 높은 그룹
            basket = factor_filtered[factor_filtered > high_th]

        illiq        = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
        threshold    = illiq.quantile(1 - Amihud_threshold_temp)
        illiquid_top = illiq[illiq >= threshold].index

        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket.index)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1/len(basket), index=basket.index)
        else:  # Cap
            cap_seg = mkt_cap.loc[start_date, basket.index]
            target_weights = cap_seg / cap_seg.sum()

        all_index     = target_weights.index.union(prev_weights.index)
        target_w      = target_weights.reindex(all_index, fill_value=0)
        prev_w        = prev_weights.reindex(all_index, fill_value=0)
        delta_w       = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate     = np.where(delta_w.index.isin(illiquid_top), high_cost, low_cost)
        trade_cost    = (trade_amounts * cost_rate).sum()
        NAV_new       = NAV - trade_cost

        current_portfolio_value = target_weights * NAV_new
        ret_seg = ret_wins.loc[end_date, basket.index]
        next_portfolio_value = current_portfolio_value * (ret_seg + 1)

        NAV_new       = next_portfolio_value.sum()
        portfolio_ret = NAV_new / NAV - 1

        prev_portfolio = next_portfolio_value
        NAV            = NAV_new

        total_trade.loc[start_date]    = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    return portfolio_return, total_trade

##### **계산**

In [73]:
cap_cut           = 2                                # 포트폴리오를 나누는 분위 수 e.g. 5 → 5분위 
c                 = [0, 1]               

weight_method     = ['Equal', 'Cap']                 # ['Equal', Cap']

cost              = [(0.008, 0.003), (0, 0)]         # (high_cost, low_cost)            e.g. [(0.008, 0.003), (0, 0)]
                                                     
wins_threshold    = [0.01]                           # 수익률 윈저라이징                  e.g. [0.01, 0.05]
trading_threshold = [0.1]                            # 60일 평균 거래대금 필터링           e.g. [0.1, 0.0]
Amihud_threshold  = [0.2]                            # 유동성 기준                       e.g. [0.2, 0.0]

initial_NAV       = 1                                # 초기값

In [74]:
trading_threshold_temp = trading_threshold[0]
weight_method_temp     = weight_method[0]
cost_temp              = cost[0]
wins_threshold_temp    = wins_threshold[0]
Amihud_threshold_temp  = Amihud_threshold[0]

In [75]:
th = 'long'
results_long = Parallel(n_jobs=-1)(
    delayed(run_strategy)(
        c_temp, th,
        wins_threshold_temp, weight_method_temp, cost_temp,
        trading_threshold_temp, Amihud_threshold_temp,
        initial_NAV
    )
    for c_temp in c
)

th = 'short'
results_short = Parallel(n_jobs=-1)(
    delayed(run_strategy)(
        c_temp, th,
        wins_threshold_temp, weight_method_temp, cost_temp,
        trading_threshold_temp, Amihud_threshold_temp,
        initial_NAV
    )
    for c_temp in c
)

factor_portfolio_df = pd.DataFrame()

# Long 결과 추가
for (portfolio_return, total_trade) in results_long:
    factor_portfolio_df[portfolio_return.name] = portfolio_return

# Short 결과 추가
for (portfolio_return, total_trade) in results_short:
    factor_portfolio_df[portfolio_return.name] = portfolio_return

In [76]:
factor_portfolio_df.tail()

,C(1/2)_Q(long),C(2/2)_Q(long),C(1/2)_Q(short),C(2/2)_Q(short)
2025-06-30,0.028779,0.056202,0.024553,0.071289
2025-07-31,0.010412,0.029450,-0.022716,-0.011194
2025-08-31,-0.016970,-0.029710,-0.038228,-0.007609
2025-09-30,-0.003239,0.016120,0.012890,0.053439
2025-10-31,-0.022113,0.014695,-0.026575,0.055429


In [77]:
# long-short 스프레드 계산
small_ls = factor_portfolio_df["C(1/2)_Q(long)"] - factor_portfolio_df["C(1/2)_Q(short)"]
big_ls   = factor_portfolio_df["C(2/2)_Q(long)"] - factor_portfolio_df["C(2/2)_Q(short)"]

# 두 그룹 평균 → 최종 factor return
factor_return = (small_ls + big_ls) / 2
factor_return.name = "Factor"

##### **Output : ivol 팩터 포트폴리오** 

In [78]:
factor_return

2000-02-29   -0.214391
2000-03-31    0.024671
2000-04-30    0.023216
2000-05-31    0.012540
2000-06-30    0.065368
                ...   
2025-06-30   -0.005430
2025-07-31    0.036886
2025-08-31   -0.000421
2025-09-30   -0.026724
2025-10-31   -0.018136
Name: Factor, Length: 309, dtype: float64

##### **회귀분석**

##### **Code**

In [79]:
# 1. 두 데이터 공통 구간 맞추기
df = pd.concat([portfolio, factors], axis=1, join="inner").dropna()

In [80]:
# 2. 종속변수: 포트폴리오 초과수익률
y = df['Return'] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM']
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})  # Newey-West 표준오차

##### **기존 결과**

In [81]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.899
Model:                            OLS   Adj. R-squared:                  0.898
Method:                 Least Squares   F-statistic:                     414.4
Date:                Fri, 14 Nov 2025   Prob (F-statistic):          1.09e-121
Time:                        13:36:02   Log-Likelihood:                 780.18
No. Observations:                 309   AIC:                            -1550.
Df Residuals:                     304   BIC:                            -1532.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0018      0.001      1.546      0.1

---

##### **Code**

In [82]:
# 1. 두 데이터 공통 구간 맞추기
df = pd.concat([portfolio, factors], axis=1, join="inner").dropna()

# factor_return 추가
df["Factor"] = factor_return.loc[df.index]

# 2. 종속변수: 포트폴리오 초과수익률
y = df['Return'] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM, MyFactor
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM'],
    "Factor": df["Factor"]
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})

##### **새로운 팩터를 포함한 결과**

In [83]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.928
Model:                            OLS   Adj. R-squared:                  0.927
Method:                 Least Squares   F-statistic:                     335.1
Date:                Fri, 14 Nov 2025   Prob (F-statistic):          3.92e-121
Time:                        13:36:02   Log-Likelihood:                 832.40
No. Observations:                 309   AIC:                            -1653.
Df Residuals:                     303   BIC:                            -1630.
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0045      0.001     -3.632      0.0

---
## **5. 종목 수**

In [84]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

subset = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

In [85]:
##### **선택된 포트폴리오 통계 계산 (월별 수익률 기준)**

import numpy as np

def calculate_portfolio_statistics(portfolio_returns, benchmark_return=None):
    """
    포트폴리오 통계 계산 함수 - 월별 수익률 기준
    
    Parameters:
    -----------
    portfolio_returns : Series or DataFrame
        포트폴리오 수익률 시계열 (Series) 또는 여러 포트폴리오 (DataFrame)
        월별 수익률 데이터를 가정
    benchmark_return : Series, optional
        벤치마크 수익률 (예: 시장 수익률) - 월별 데이터여야 함
    
    Returns:
    --------
    stats : dict or DataFrame
        포트폴리오 통계 정보
    """
    # DataFrame인 경우 각 컬럼별로 처리
    if isinstance(portfolio_returns, pd.DataFrame):
        all_stats = {}
        for col in portfolio_returns.columns:
            stats, _, _ = calculate_portfolio_statistics(portfolio_returns[col], benchmark_return)
            all_stats[col] = stats
        return pd.DataFrame(all_stats)
    
    # Series 처리
    returns = portfolio_returns.dropna()
    if len(returns) == 0:
        return {}
    
    # 누적 수익률 계산
    cum_return = (1 + returns).cumprod() - 1
    
    # 기본 통계 (월별 데이터 기준: 12개월로 연화)
    stats = {}
    stats['총 수익률'] = cum_return.iloc[-1] if len(cum_return) > 0 else np.nan
    stats['연평균 수익률'] = (1 + stats['총 수익률']) ** (12 / len(returns)) - 1 if len(returns) > 0 and stats['총 수익률'] > -1 else np.nan
    stats['월평균 수익률'] = returns.mean()
    stats['월별 수익률 표준편차'] = returns.std()
    stats['연화 표준편차'] = stats['월별 수익률 표준편차'] * np.sqrt(12) if not pd.isna(stats['월별 수익률 표준편차']) else np.nan
    stats['샤프 비율'] = (stats['연평균 수익률'] / stats['연화 표준편차']) if (not pd.isna(stats['연평균 수익률']) and not pd.isna(stats['연화 표준편차']) and stats['연화 표준편차'] > 0) else np.nan
    
    # 최대 낙폭 (Maximum Drawdown)
    if len(cum_return) > 0:
        running_max = cum_return.expanding().max()
        drawdown = cum_return - running_max
        stats['최대 낙폭'] = drawdown.min()
    else:
        stats['최대 낙폭'] = np.nan
    
    # 승률 (양수 수익률 비율)
    stats['승률'] = (returns > 0).sum() / len(returns)
    
    # 벤치마크 대비 초과수익률
    if benchmark_return is not None:
        # 인덱스 정렬
        common_idx = returns.index.intersection(benchmark_return.index)
        if len(common_idx) > 0:
            returns_aligned = returns.loc[common_idx]
            benchmark_aligned = benchmark_return.loc[common_idx]
            excess_return = returns_aligned - benchmark_aligned
            stats['초과수익률 (월평균)'] = excess_return.mean()
            stats['초과수익률 (연평균)'] = stats['초과수익률 (월평균)'] * 12 if not pd.isna(stats['초과수익률 (월평균)']) else np.nan
            
            # 정보 비율 (Information Ratio)
            excess_std = excess_return.std()
            stats['정보 비율'] = (stats['초과수익률 (연평균)'] / (excess_std * np.sqrt(12))) if (not pd.isna(excess_std) and excess_std > 0 and not pd.isna(stats['초과수익률 (연평균)'])) else np.nan
    
    return stats, returns, cum_return

# 벤치마크 설정 (KOSPI 수익률 사용 - 월별로 리샘플링 필요)
benchmark = None
if 'KOSPI' in factors.columns:
    # 일별 KOSPI를 월말 기준으로 리샘플링
    kospi_monthly = factors['KOSPI'].resample('ME').last()
    benchmark = kospi_monthly.loc[subset.index] if len(subset) > 0 else None

# 선택된 포트폴리오 통계 계산
print("=" * 60)
print("선택된 포트폴리오 통계 분석 (월별 수익률 기준)")
print("=" * 60)
print(f"포트폴리오 개수: {subset.shape[1]}")
print(f"기간: {subset.index.min().date()} ~ {subset.index.max().date()}")
print(f"개월 수: {len(subset)}\n")

# 각 포트폴리오별 통계 계산
portfolio_stats = {}
for col in subset.columns:
    stats, ret_series, cum_ret = calculate_portfolio_statistics(subset[col], benchmark)
    portfolio_stats[col] = stats
    portfolio_stats[col]['returns'] = ret_series
    portfolio_stats[col]['cumulative'] = cum_ret

# 통계를 DataFrame으로 변환하여 출력
stats_summary = pd.DataFrame({
    col: {
        '총 수익률 (%)': stats['총 수익률'] * 100,
        '연평균 수익률 (%)': stats['연평균 수익률'] * 100,
        '연화 표준편차 (%)': stats['연화 표준편차'] * 100,
        '샤프 비율': stats['샤프 비율'],
        '최대 낙폭 (%)': stats['최대 낙폭'] * 100,
        '승률 (%)': stats['승률'] * 100,
    }
    for col, stats in portfolio_stats.items()
})

# 벤치마크 대비 초과수익률 추가
if benchmark is not None:
    for col in stats_summary.columns:
        if '초과수익률 (연평균)' in portfolio_stats[col]:
            stats_summary.loc['벤치마크 대비 초과수익률 (%)', col] = portfolio_stats[col]['초과수익률 (연평균)'] * 100
        if '정보 비율' in portfolio_stats[col]:
            stats_summary.loc['정보 비율', col] = portfolio_stats[col]['정보 비율']

print("\n[포트폴리오 통계 요약]\n")
print(stats_summary.round(2))

# 상세 통계 출력
print("\n[포트폴리오별 상세 통계]\n")
for col, stats in portfolio_stats.items():
    print(f"--- {col} ---")
    print(f"  총 수익률: {stats['총 수익률']*100:.2f}%")
    print(f"  연평균 수익률: {stats['연평균 수익률']*100:.2f}%")
    print(f"  연화 표준편차: {stats['연화 표준편차']*100:.2f}%")
    print(f"  샤프 비율: {stats['샤프 비율']:.3f}")
    print(f"  최대 낙폭: {stats['최대 낙폭']*100:.2f}%")
    print(f"  승률: {stats['승률']*100:.2f}%")
    if '초과수익률 (연평균)' in stats and not pd.isna(stats['초과수익률 (연평균)']):
        print(f"  벤치마크 대비 초과수익률: {stats['초과수익률 (연평균)']*100:.2f}%")
    if '정보 비율' in stats and not pd.isna(stats['정보 비율']):
        print(f"  정보 비율: {stats['정보 비율']:.3f}")
    print()

선택된 포트폴리오 통계 분석 (월별 수익률 기준)
포트폴리오 개수: 5
기간: 2000-02-29 ~ 2025-10-31
개월 수: 309


[포트폴리오 통계 요약]

                   Q(1/5)  Q(2/5)  Q(3/5)  Q(4/5)  Q(5/5)
총 수익률 (%)          216.36  112.85  138.37  -53.26  -99.85
연평균 수익률 (%)          4.57    2.98    3.43   -2.91  -22.27
연화 표준편차 (%)         20.01   22.57   25.04   26.42   29.74
샤프 비율                0.23    0.13    0.14   -0.11   -0.75
최대 낙폭 (%)          -79.14 -131.24 -108.65  -92.68 -128.98
승률 (%)              55.34   53.07   53.72   54.05   43.37
벤치마크 대비 초과수익률 (%)   -1.55   -2.55   -1.49   -7.40  -28.36
정보 비율               -0.18   -0.31   -0.14   -0.51   -1.30

[포트폴리오별 상세 통계]

--- Q(1/5) ---
  총 수익률: 216.36%
  연평균 수익률: 4.57%
  연화 표준편차: 20.01%
  샤프 비율: 0.229
  최대 낙폭: -79.14%
  승률: 55.34%
  벤치마크 대비 초과수익률: -1.55%
  정보 비율: -0.184

--- Q(2/5) ---
  총 수익률: 112.85%
  연평균 수익률: 2.98%
  연화 표준편차: 22.57%
  샤프 비율: 0.132
  최대 낙폭: -131.24%
  승률: 53.07%
  벤치마크 대비 초과수익률: -2.55%
  정보 비율: -0.307

--- Q(3/5) ---
  총 수익률: 138.37%
  연평균 수익률: 3.43%
  연화 표준편차:

In [86]:
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np

# 함수 정의
def run_strategy(wins_threshold_temp, weight_method_temp, cost_temp,
                 trading_threshold_temp, Amihud_threshold_temp, n_temp):
    """
    n_temp: 이제 분위수가 아닌 선택할 종목 개수 (상위 또는 하위)
    """
    high_cost, low_cost = cost_temp
    NAV = initial_NAV

    # n_temp를 종목 개수로 사용
    num_stocks = n_temp
    
    portfolio_name   = f"{weight_method_temp}_R{wins_threshold_temp}_H{int(high_cost*10000)}L{int(low_cost*10000)}_T{trading_threshold_temp}_A{Amihud_threshold_temp}_N{num_stocks}"
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade      = pd.Series(dtype=float, name=portfolio_name)

    # 수익률 윈저라이징
    ret_wins = ret_m.clip(
        lower = ret_m.quantile(wins_threshold_temp),
        upper = ret_m.quantile(1 - wins_threshold_temp),
        axis  = 1
    )

    prev_portfolio = None
    
    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        # 거래대금 하위 x% 필터링
        series          = trading_value_60.loc[start_date]
        threshold       = series.quantile(trading_threshold_temp)
        filtered        = series[series > threshold]
        factor_filtered = factor_df.loc[start_date, filtered.index]

        # 종목 선정: MAX가 낮은 N개 종목 선택 (nsmallest 사용)
        if len(factor_filtered) < num_stocks:
            # 종목 수가 부족한 경우 사용 가능한 모든 종목 사용
            basket = factor_filtered.copy()
        else:
            # MAX가 낮은 N개 종목 선택 (Low-MAX 전략)
            basket = factor_filtered.nsmallest(num_stocks)

        if basket.empty:
            continue

        # Amihud illiquidity 계산
        illiq        = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
        threshold    = illiq.quantile(1 - Amihud_threshold_temp)
        illiquid_top = illiq[illiq >= threshold].index

        # 이전 비중
        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket.index)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        # 목표 비중
        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1/len(basket), index=basket.index)
        else:  # Cap
            cap_seg = mkt_cap.loc[start_date, basket.index]
            target_weights = cap_seg / cap_seg.sum()

        # 거래비용 반영
        all_index     = target_weights.index.union(prev_weights.index)
        target_w      = target_weights.reindex(all_index, fill_value=0)
        prev_w        = prev_weights.reindex(all_index, fill_value=0)
        delta_w       = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate     = np.where(delta_w.index.isin(illiquid_top), high_cost, low_cost)
        trade_cost    = (trade_amounts * cost_rate).sum()
        NAV_new       = NAV - trade_cost

        # 포트폴리오 가치
        current_portfolio_value = target_weights * NAV_new
        ret_seg                 = ret_wins.loc[end_date, basket.index]
        next_portfolio_value    = current_portfolio_value * (ret_seg + 1)

        NAV_new       = next_portfolio_value.sum()
        portfolio_ret = NAV_new / NAV - 1

        prev_portfolio = next_portfolio_value
        NAV            = NAV_new

        total_trade.loc[start_date]    = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    return portfolio_return, total_trade

# 포트폴리오 선택 함수
def select_columns(df, fixed_options):
    selected = []
    new_names = {}
    
    for c in df.columns:
        parts = c.split('_')
        
        # 조건 체크 (고정한 옵션은 정확히 일치해야 함)
        if all(parts[i] == v or v == "*" for i, v in fixed_options.items()):
            selected.append(c)
            
            # 새 이름은 "고정하지 않은 옵션만"
            free_parts = [parts[i] for i, v in fixed_options.items() if v == "*"]
            new_name = "_".join(free_parts)
            new_names[c] = new_name
    
    # 선택된 subset 반환, 컬럼명은 자유 요소만
    subset = df[selected].rename(columns=new_names)
    return subset

In [87]:
# 팩터 설정
factor_df         = MAX5_df

In [88]:
start_point       = '2000-01-01'                     # 백테스트 기간 설정
end_point         = '2025-10-31'

q_cut             = 5                                # 포트폴리오를 나누는 분위 수 e.g. 5 → 5분위 
n                 = [20, 50, 75, 100, 200, 300, 400, 500, 1000]                  # 0 : 하위 ~ (q_cut - 1) : 상위    e.g. [0, 1, 2, 3, 4]

weight_method     = ['Equal', 'Cap']                 # ['Equal', Cap']

cost              = [(0.008, 0.003)]         # (high_cost, low_cost)            e.g. [(0.008, 0.003), (0, 0)]
                                                     
wins_threshold    = [0.01]                     # 수익률 윈저라이징                 e.g. [0.01, 0.05]
trading_threshold = [0.1]                       # 60일 평균 거래대금 필터링         e.g. [0.1, 0.0]
Amihud_threshold  = [0.2]                       # 유동성 기준                       e.g. [0.2, 0.0]

initial_NAV       = 1                                # 초기값

In [89]:
# ret_m        = total_adj_close.resample('QE').last().ffill().pct_change() # 분기 리밸런싱
ret_m        = total_adj_close.resample('ME').last().ffill().pct_change()

portfolio_df_1 = pd.DataFrame()
trade_df_1     = pd.DataFrame()

month_ends   = factor_df.resample('ME').last().index
month_ends   = month_ends[(month_ends >= pd.to_datetime(start_point)) &
                        (month_ends <= pd.to_datetime(end_point))]

C:\Users\XH58\AppData\Local\Temp\ipykernel_11812\323625349.py:2: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [90]:
# 모든 조합 만들기
param_list = [
    (wins_threshold_temp, weight_method_temp, cost_temp, trading_threshold_temp, Amihud_threshold_temp, n_temp)
    for wins_threshold_temp in wins_threshold
    for weight_method_temp in weight_method
    for cost_temp in cost
    for trading_threshold_temp in trading_threshold
    for Amihud_threshold_temp in Amihud_threshold
    for n_temp in n
]

# 병렬 실행
results = Parallel(n_jobs=-1)(
    delayed(run_strategy)(*params) for params in tqdm(param_list)
)

# 결과 합치기
portfolio_df_1 = pd.DataFrame({r[0].name: r[0] for r in results})
trade_df_1     = pd.DataFrame({r[1].name: r[1] for r in results})

100%|██████████| 18/18 [00:22<00:00,  1.26s/it]


In [91]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

subset = select_columns(
    portfolio_df_1,
    {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

In [92]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 데이터 준비
nav = (subset + 1).cumprod()
log_cum = np.log1p(subset).cumsum()

# 서브플롯 (가로 2개)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Cumulative NAV", "Log Cumulative Return"])

# 첫 번째 그래프
for col in nav.columns:
    fig.add_trace(
        go.Scatter(x=nav.index, y=nav[col], mode='lines', name=col,
                   legendgroup=col, showlegend=True),
        row=1, col=1
    )

# 두 번째 그래프
for col in log_cum.columns:
    fig.add_trace(
        go.Scatter(x=log_cum.index, y=log_cum[col], mode='lines', name=col,
                   legendgroup=col, showlegend=False),
        row=1, col=2
    )

# 레이아웃 기본 설정
fig.update_layout(
    title="Portfolio Performance Comparison",
    height=500, width=1000,
    template="plotly_white"
)

fig.update_layout(
    shapes=[
        # 왼쪽 그래프 영역 (x=0~0.45, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.0, y0=0.0, x1=0.45, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
        
        # 오른쪽 그래프 영역 (x=0.55~1, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.55, y0=0.0, x1=1.0, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
    ]
)

In [93]:
factors        = pd.read_csv("./input/factors_monthly.csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)
portfolio      = portfolio_df_1['Equal_R0.01_H80L30_T0.1_A0.2_N500']
portfolio.name = 'Return'

In [94]:
# 1. 두 데이터 공통 구간 맞추기
df = pd.concat([portfolio, factors], axis=1, join="inner").dropna()

# 2. 종속변수: 포트폴리오 초과수익률
y = df['Return'] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM']
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀 (Newey-West 표준오차)
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})

# 5. 결과 출력
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.913
Model:                            OLS   Adj. R-squared:                  0.912
Method:                 Least Squares   F-statistic:                     308.6
Date:                Fri, 14 Nov 2025   Prob (F-statistic):          1.13e-105
Time:                        13:36:40   Log-Likelihood:                 787.01
No. Observations:                 309   AIC:                            -1564.
Df Residuals:                     304   BIC:                            -1545.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0027      0.001      2.217      0.0

In [95]:
##### **선택된 포트폴리오 통계 계산 (월별 수익률 기준)**

import numpy as np

def calculate_portfolio_statistics(portfolio_returns, benchmark_return=None):
    """
    포트폴리오 통계 계산 함수 - 월별 수익률 기준
    
    Parameters:
    -----------
    portfolio_returns : Series or DataFrame
        포트폴리오 수익률 시계열 (Series) 또는 여러 포트폴리오 (DataFrame)
        월별 수익률 데이터를 가정
    benchmark_return : Series, optional
        벤치마크 수익률 (예: 시장 수익률) - 월별 데이터여야 함
    
    Returns:
    --------
    stats : dict or DataFrame
        포트폴리오 통계 정보
    """
    # DataFrame인 경우 각 컬럼별로 처리
    if isinstance(portfolio_returns, pd.DataFrame):
        all_stats = {}
        for col in portfolio_returns.columns:
            stats, _, _ = calculate_portfolio_statistics(portfolio_returns[col], benchmark_return)
            all_stats[col] = stats
        return pd.DataFrame(all_stats)
    
    # Series 처리
    returns = portfolio_returns.dropna()
    if len(returns) == 0:
        return {}
    
    # 누적 수익률 계산
    cum_return = (1 + returns).cumprod() - 1
    
    # 기본 통계 (월별 데이터 기준: 12개월로 연화)
    stats = {}
    stats['총 수익률'] = cum_return.iloc[-1] if len(cum_return) > 0 else np.nan
    stats['연평균 수익률'] = (1 + stats['총 수익률']) ** (12 / len(returns)) - 1 if len(returns) > 0 and stats['총 수익률'] > -1 else np.nan
    stats['월평균 수익률'] = returns.mean()
    stats['월별 수익률 표준편차'] = returns.std()
    stats['연화 표준편차'] = stats['월별 수익률 표준편차'] * np.sqrt(12) if not pd.isna(stats['월별 수익률 표준편차']) else np.nan
    stats['샤프 비율'] = (stats['연평균 수익률'] / stats['연화 표준편차']) if (not pd.isna(stats['연평균 수익률']) and not pd.isna(stats['연화 표준편차']) and stats['연화 표준편차'] > 0) else np.nan
    
    # 최대 낙폭 (Maximum Drawdown)
    if len(cum_return) > 0:
        running_max = cum_return.expanding().max()
        drawdown = cum_return - running_max
        stats['최대 낙폭'] = drawdown.min()
    else:
        stats['최대 낙폭'] = np.nan
    
    # 승률 (양수 수익률 비율)
    stats['승률'] = (returns > 0).sum() / len(returns)
    
    # 벤치마크 대비 초과수익률
    if benchmark_return is not None:
        # 인덱스 정렬
        common_idx = returns.index.intersection(benchmark_return.index)
        if len(common_idx) > 0:
            returns_aligned = returns.loc[common_idx]
            benchmark_aligned = benchmark_return.loc[common_idx]
            excess_return = returns_aligned - benchmark_aligned
            stats['초과수익률 (월평균)'] = excess_return.mean()
            stats['초과수익률 (연평균)'] = stats['초과수익률 (월평균)'] * 12 if not pd.isna(stats['초과수익률 (월평균)']) else np.nan
            
            # 정보 비율 (Information Ratio)
            excess_std = excess_return.std()
            stats['정보 비율'] = (stats['초과수익률 (연평균)'] / (excess_std * np.sqrt(12))) if (not pd.isna(excess_std) and excess_std > 0 and not pd.isna(stats['초과수익률 (연평균)'])) else np.nan
    
    return stats, returns, cum_return

# 벤치마크 설정 (KOSPI 수익률 사용 - 월별로 리샘플링 필요)
benchmark = None
if 'KOSPI' in factors.columns:
    # 일별 KOSPI를 월말 기준으로 리샘플링
    kospi_monthly = factors['KOSPI'].resample('ME').last()
    benchmark = kospi_monthly.loc[subset.index] if len(subset) > 0 else None

# 선택된 포트폴리오 통계 계산
print("=" * 60)
print("선택된 포트폴리오 통계 분석 (월별 수익률 기준)")
print("=" * 60)
print(f"포트폴리오 개수: {subset.shape[1]}")
print(f"기간: {subset.index.min().date()} ~ {subset.index.max().date()}")
print(f"개월 수: {len(subset)}\n")

# 각 포트폴리오별 통계 계산
portfolio_stats = {}
for col in subset.columns:
    stats, ret_series, cum_ret = calculate_portfolio_statistics(subset[col], benchmark)
    portfolio_stats[col] = stats
    portfolio_stats[col]['returns'] = ret_series
    portfolio_stats[col]['cumulative'] = cum_ret

# 통계를 DataFrame으로 변환하여 출력
stats_summary = pd.DataFrame({
    col: {
        '총 수익률 (%)': stats['총 수익률'] * 100,
        '연평균 수익률 (%)': stats['연평균 수익률'] * 100,
        '연화 표준편차 (%)': stats['연화 표준편차'] * 100,
        '샤프 비율': stats['샤프 비율'],
        '최대 낙폭 (%)': stats['최대 낙폭'] * 100,
        '승률 (%)': stats['승률'] * 100,
    }
    for col, stats in portfolio_stats.items()
})

# 벤치마크 대비 초과수익률 추가
if benchmark is not None:
    for col in stats_summary.columns:
        if '초과수익률 (연평균)' in portfolio_stats[col]:
            stats_summary.loc['벤치마크 대비 초과수익률 (%)', col] = portfolio_stats[col]['초과수익률 (연평균)'] * 100
        if '정보 비율' in portfolio_stats[col]:
            stats_summary.loc['정보 비율', col] = portfolio_stats[col]['정보 비율']

print("\n[포트폴리오 통계 요약]\n")
print(stats_summary.round(2))

# 상세 통계 출력
print("\n[포트폴리오별 상세 통계]\n")
for col, stats in portfolio_stats.items():
    print(f"--- {col} ---")
    print(f"  총 수익률: {stats['총 수익률']*100:.2f}%")
    print(f"  연평균 수익률: {stats['연평균 수익률']*100:.2f}%")
    print(f"  연화 표준편차: {stats['연화 표준편차']*100:.2f}%")
    print(f"  샤프 비율: {stats['샤프 비율']:.3f}")
    print(f"  최대 낙폭: {stats['최대 낙폭']*100:.2f}%")
    print(f"  승률: {stats['승률']*100:.2f}%")
    if '초과수익률 (연평균)' in stats and not pd.isna(stats['초과수익률 (연평균)']):
        print(f"  벤치마크 대비 초과수익률: {stats['초과수익률 (연평균)']*100:.2f}%")
    if '정보 비율' in stats and not pd.isna(stats['정보 비율']):
        print(f"  정보 비율: {stats['정보 비율']:.3f}")
    print()

선택된 포트폴리오 통계 분석 (월별 수익률 기준)
포트폴리오 개수: 9
기간: 2000-02-29 ~ 2025-10-31
개월 수: 309


[포트폴리오 통계 요약]

                      N20     N50     N75    N100    N200    N300    N400  \
총 수익률 (%)          -29.13  138.08  272.53  442.70  732.24  806.53  961.96   
연평균 수익률 (%)         -1.33    3.43    5.24    6.79    8.58    8.94    9.61   
연화 표준편차 (%)         15.44   17.20   18.30   18.88   20.38   21.27   21.80   
샤프 비율               -0.09    0.20    0.29    0.36    0.42    0.42    0.44   
최대 낙폭 (%)         -194.23 -167.77 -186.11 -252.15 -283.87 -303.95 -355.35   
승률 (%)              46.28   51.46   52.43   53.72   55.02   55.34   55.99   
벤치마크 대비 초과수익률 (%)   -8.17   -3.16   -1.23    0.34    2.30    2.82    3.55   
정보 비율               -0.50   -0.23   -0.09    0.03    0.17    0.22    0.27   

                      N500   N1000  
총 수익률 (%)          1071.43  941.12  
연평균 수익률 (%)          10.03    9.53  
연화 표준편차 (%)          22.31   23.96  
샤프 비율                 0.45    0.40  
최대 낙폭 (%)          -387.89

---
## **6. Short**

In [96]:
q_cut             = 5                                # 포트폴리오를 나누는 분위 수 e.g. 5 → 5분위 
n                 = [0, 1, 2, 3, 4]                  # 0 : 하위 ~ (q_cut - 1) : 상위    e.g. [0, 1, 2, 3, 4]

In [97]:
def run_short_strategy(c_temp, n_temp,
                 wins_threshold_temp, weight_method_temp, cost_temp,
                 trading_threshold_temp, Amihud_threshold_temp, 
                 initial_NAV=1):
    high_cost, low_cost = cost_temp
    NAV = initial_NAV
    portfolio_name = f"C({c_temp+1}/{cap_cut})_Q({n_temp+1}/{q_cut})"
    
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade      = pd.Series(dtype=float, name=portfolio_name)
    
    # 수익률 윈저라이징
    ret_wins = ret_m.clip(
        lower = ret_m.quantile(wins_threshold_temp),
        upper = ret_m.quantile(1 - wins_threshold_temp),
        axis  = 1
    )
    
    prev_portfolio = None

    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        cap_series    = mkt_cap.loc[start_date]
        cap_quantiles = pd.qcut(cap_series, q=cap_cut, labels=False)
        cap_filtered  = cap_series[cap_quantiles == c_temp]

        series    = trading_value_60.loc[start_date, cap_filtered.index]
        threshold = series.quantile(trading_threshold_temp)
        filtered  = series[series > threshold]
        factor_filtered = factor_df.loc[start_date, filtered.index]

        quantiles = pd.qcut(factor_filtered, q=q_cut, labels=False)
        basket    = factor_filtered[quantiles == n_temp]
        if basket.empty:
            continue

        illiq        = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
        threshold    = illiq.quantile(1 - Amihud_threshold_temp)
        illiquid_top = illiq[illiq >= threshold].index

        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket.index)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1/len(basket), index=basket.index)
        else:  # Cap
            cap_seg = mkt_cap.loc[start_date, basket.index]
            target_weights = cap_seg / cap_seg.sum()

        all_index     = target_weights.index.union(prev_weights.index)
        target_w      = target_weights.reindex(all_index, fill_value=0)
        prev_w        = prev_weights.reindex(all_index, fill_value=0)
        delta_w       = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate     = np.where(delta_w.index.isin(illiquid_top), high_cost, low_cost)
        trade_cost    = (trade_amounts * cost_rate).sum()
        NAV_new       = NAV - trade_cost

        current_portfolio_value = target_weights * NAV_new
        ret_seg = ret_wins.loc[end_date, basket.index]
        # 숏 전략: 수익률 반전
        next_portfolio_value = current_portfolio_value * (1 - ret_seg)

        NAV_new       = next_portfolio_value.sum()
        portfolio_ret = NAV_new / NAV - 1

        prev_portfolio = next_portfolio_value
        NAV            = NAV_new

        total_trade.loc[start_date]    = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    return portfolio_return, total_trade


##### **계산 실행**

In [98]:
cap_cut           = 5                                # 포트폴리오를 나누는 분위 수 e.g. 5 → 5분위 
c                 = [0, 1, 2, 3, 4]    

In [99]:
trading_threshold_temp = trading_threshold[0]
weight_method_temp     = weight_method[0]
cost_temp              = cost[0]
wins_threshold_temp    = wins_threshold[0]
Amihud_threshold_temp  = Amihud_threshold[0]

In [100]:
# --- 병렬 실행 ---
results = Parallel(n_jobs=-1)(
    delayed(run_short_strategy)(
        c_temp, n_temp,
        wins_threshold_temp, weight_method_temp, cost_temp,
        trading_threshold_temp, Amihud_threshold_temp,
        initial_NAV
    )
    for c_temp in c
    for n_temp in n
)

# --- 결과 합치기 ---
cap_cut_short_profolio_df = pd.DataFrame()
for (portfolio_return, total_trade) in results:
    cap_cut_short_profolio_df[portfolio_return.name] = portfolio_return

##### **Output**

In [101]:
cap_cut_short_profolio_df.tail()

,C(1/5)_Q(1/5),C(1/5)_Q(2/5),C(1/5)_Q(3/5),C(1/5)_Q(4/5),C(1/5)_Q(5/5),C(2/5)_Q(1/5),C(2/5)_Q(2/5),C(2/5)_Q(3/5),C(2/5)_Q(4/5),C(2/5)_Q(5/5),...,C(4/5)_Q(1/5),C(4/5)_Q(2/5),C(4/5)_Q(3/5),C(4/5)_Q(4/5),C(4/5)_Q(5/5),C(5/5)_Q(1/5),C(5/5)_Q(2/5),C(5/5)_Q(3/5),C(5/5)_Q(4/5),C(5/5)_Q(5/5)
2025-06-30,-0.008973,-0.050845,-0.040536,-0.051269,-0.016095,-0.036877,-0.072172,-0.058073,-0.070491,-0.045215,...,-0.055772,-0.051748,-0.073554,-0.067735,-0.059326,-0.091698,-0.092939,-0.117773,-0.111450,-0.120169
2025-07-31,-0.006898,-0.019402,-0.032908,-0.012523,0.010195,-0.022246,-0.033643,-0.016357,-0.016713,0.011714,...,-0.037039,-0.039479,-0.038564,-0.026460,0.024719,-0.053799,-0.049697,-0.032197,-0.040194,0.021287
2025-08-31,0.001480,-0.008484,-0.014554,-0.001141,0.019402,0.004508,0.006996,0.008379,0.024721,0.026516,...,0.021004,0.014453,0.015870,0.020172,0.013110,0.022342,0.014259,0.018305,-0.006045,-0.018433
2025-09-30,-0.012085,-0.017706,-0.008379,-0.029109,-0.038972,-0.001038,0.000874,-0.022245,-0.040413,-0.004338,...,-0.006786,-0.065886,-0.060003,-0.049967,-0.061311,-0.022522,-0.060653,-0.037666,-0.069627,-0.078736
2025-10-31,-0.004985,0.010535,0.016528,0.008752,0.019479,0.017270,0.005092,0.024115,0.022171,0.033950,...,-0.010831,-0.008338,-0.010444,-0.021840,-0.057760,-0.046922,-0.119954,-0.056949,-0.112815,-0.079506


---
##### **Plot**

##### **Code**

In [102]:
def CAGR(series, periods_per_year=12):
    """월별 수익률 시리즈 기준 CAGR 계산"""
    series = series.dropna()
    if series.empty:
        return np.nan
    cumulative = (1 + series).cumprod()
    start, end = cumulative.index[0], cumulative.index[-1]
    n_years = (end - start).days / 365.25
    return (cumulative.iloc[-1] ** (1 / n_years)) - 1

In [103]:
# heatmap용 DF 초기화
cap_cut = 5
q_cut   = 5
cagr_matrix = pd.DataFrame(index=[f"C({i+1}/{cap_cut})" for i in range(cap_cut)],
                           columns=[f"Q({j+1}/{q_cut})" for j in range(q_cut)])

for col in cap_cut_short_profolio_df.columns:
    # 이름 parsing: "C(i/5)_Q(j/5)"
    c_part, q_part = col.split('_')
    cagr_matrix.loc[c_part, q_part] = CAGR(cap_cut_short_profolio_df[col])
    
cagr_matrix = cagr_matrix.astype(float)

In [104]:
import plotly.express as px

fig = px.imshow(
    cagr_matrix.astype(float),
    text_auto=".1%",
    color_continuous_scale="Blues",  # 파란 계열
    aspect="equal",                  # 정사각형 셀
    labels=dict(
        x="Factor Quintile (Q)", 
        y="Market Cap Quintile (C)", 
        color="CAGR"
    )
)

fig.update_layout(
    title="CAGR Heatmap by Cap vs Factor Quantile",
    xaxis=dict(side="top"),
    width=600,   # 가로 크기
    height=600,  # 세로 크기
    margin=dict(l=60, r=60, t=80, b=60)  # 여백 조정
)

fig.show()

##### **Heatmap**

In [105]:
fig.show()

**분위수 검증**

In [106]:
from joblib import Parallel, delayed
from tqdm import tqdm

# 함수 정의
def run_strategy(wins_threshold_temp, weight_method_temp, cost_temp,
                 trading_threshold_temp, Amihud_threshold_temp, n_temp):
    
    high_cost, low_cost = cost_temp
    NAV = initial_NAV

    portfolio_name   = f"{weight_method_temp}_R{wins_threshold_temp}_H{int(high_cost*10000)}L{int(low_cost*10000)}_T{trading_threshold_temp}_A{Amihud_threshold_temp}_Q({n_temp+1}/{q_cut})"
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade      = pd.Series(dtype=float, name=portfolio_name)

    # 수익률 윈저라이징
    ret_wins = ret_m.clip(
        lower = ret_m.quantile(wins_threshold_temp),
        upper = ret_m.quantile(1 - wins_threshold_temp),
        axis  = 1
    )

    prev_portfolio = None
    
    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        # 거래대금 하위 x% 필터링
        series          = trading_value_60.loc[start_date]
        threshold       = series.quantile(trading_threshold_temp)
        filtered        = series[series > threshold]
        factor_filtered = factor_df.loc[start_date, filtered.index]

        # 종목 선정 (분위수)
        quantiles = pd.qcut(factor_filtered, q=q_cut, labels=False)
        basket    = factor_filtered[quantiles == n_temp]

        if basket.empty:
            continue

        # Amihud illiquidity 계산
        illiq        = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
        threshold    = illiq.quantile(1 - Amihud_threshold_temp)
        illiquid_top = illiq[illiq >= threshold].index

        # 이전 비중
        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket.index)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        # 목표 비중
        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1/len(basket), index=basket.index)
        else:  # Cap
            cap_seg = mkt_cap.loc[start_date, basket.index]
            target_weights = cap_seg / cap_seg.sum()

        # 거래비용 반영
        all_index     = target_weights.index.union(prev_weights.index)
        target_w      = target_weights.reindex(all_index, fill_value=0)
        prev_w        = prev_weights.reindex(all_index, fill_value=0)
        delta_w       = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate     = np.where(delta_w.index.isin(illiquid_top), high_cost, low_cost)
        trade_cost    = (trade_amounts * cost_rate).sum()
        NAV_new       = NAV - trade_cost

        # 포트폴리오 가치
        current_portfolio_value = target_weights * NAV_new
        ret_seg                 = ret_wins.loc[end_date, basket.index]
        # 숏 전략: 수익률 반전
        next_portfolio_value    = current_portfolio_value * (1 - ret_seg)

        NAV_new       = next_portfolio_value.sum()
        portfolio_ret = NAV_new / NAV - 1

        prev_portfolio = next_portfolio_value
        NAV            = NAV_new

        total_trade.loc[start_date]    = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    return portfolio_return, total_trade

# 포트폴리오 선택 함수
def select_columns(df, fixed_options):
    selected = []
    new_names = {}
    
    for c in df.columns:
        parts = c.split('_')
        
        # 조건 체크 (고정한 옵션은 정확히 일치해야 함)
        if all(parts[i] == v or v == "*" for i, v in fixed_options.items()):
            selected.append(c)
            
            # 새 이름은 "고정하지 않은 옵션만"
            free_parts = [parts[i] for i, v in fixed_options.items() if v == "*"]
            new_name = "_".join(free_parts)
            new_names[c] = new_name
    
    # 선택된 subset 반환, 컬럼명은 자유 요소만
    subset = df[selected].rename(columns=new_names)
    return subset

---
##### **설정**

In [107]:
# 팩터 설정
factor_df         = MAX5_df

In [108]:
start_point       = '2000-01-01'                     # 백테스트 기간 설정
end_point         = '2025-10-31'

q_cut             = 5                                # 포트폴리오를 나누는 분위 수 e.g. 5 → 5분위 
n                 = [0, 1, 2, 3, 4]                  # 0 : 하위 ~ (q_cut - 1) : 상위    e.g. [0, 1, 2, 3, 4]

weight_method     = ['Equal', 'Cap']                 # ['Equal', Cap']

cost              = [(0.008, 0.003), (0, 0)]         # (high_cost, low_cost)            e.g. [(0.008, 0.003), (0, 0)]
                                                     
wins_threshold    = [0.01]                           # 수익률 윈저라이징                  e.g. [0.01, 0.05]
trading_threshold = [0.1]                            # 60일 평균 거래대금 필터링           e.g. [0.1, 0.0]
Amihud_threshold  = [0.2]                            # 유동성 기준                       e.g. [0.2, 0.0]

initial_NAV       = 1                                # 초기값

In [109]:
# ret_m        = total_adj_close.resample('QE').last().ffill().pct_change() # 분기 리밸런싱
ret_m        = total_adj_close.resample('ME').last().ffill().pct_change()

portfolio_df_2 = pd.DataFrame()
trade_df_2     = pd.DataFrame()

month_ends   = factor_df.resample('ME').last().index
month_ends   = month_ends[(month_ends >= pd.to_datetime(start_point)) &
                        (month_ends <= pd.to_datetime(end_point))]

C:\Users\XH58\AppData\Local\Temp\ipykernel_11812\3665085790.py:2: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [110]:
# 모든 조합 만들기
param_list = [
    (wins_threshold_temp, weight_method_temp, cost_temp, trading_threshold_temp, Amihud_threshold_temp, n_temp)
    for wins_threshold_temp in wins_threshold
    for weight_method_temp in weight_method
    for cost_temp in cost
    for trading_threshold_temp in trading_threshold
    for Amihud_threshold_temp in Amihud_threshold
    for n_temp in n
]

# 병렬 실행
results = Parallel(n_jobs=-1)(
    delayed(run_strategy)(*params) for params in tqdm(param_list)
)

# 결과 합치기
portfolio_df_2 = pd.DataFrame({r[0].name: r[0] for r in results})
trade_df_2     = pd.DataFrame({r[1].name: r[1] for r in results})

100%|██████████| 20/20 [00:20<00:00,  1.02s/it]


In [111]:
portfolio_df_2.tail()

,Equal_R0.01_H80L30_T0.1_A0.2_Q(1/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(2/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(3/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(4/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(5/5),Equal_R0.01_H0L0_T0.1_A0.2_Q(1/5),Equal_R0.01_H0L0_T0.1_A0.2_Q(2/5),Equal_R0.01_H0L0_T0.1_A0.2_Q(3/5),Equal_R0.01_H0L0_T0.1_A0.2_Q(4/5),Equal_R0.01_H0L0_T0.1_A0.2_Q(5/5),Cap_R0.01_H80L30_T0.1_A0.2_Q(1/5),Cap_R0.01_H80L30_T0.1_A0.2_Q(2/5),Cap_R0.01_H80L30_T0.1_A0.2_Q(3/5),Cap_R0.01_H80L30_T0.1_A0.2_Q(4/5),Cap_R0.01_H80L30_T0.1_A0.2_Q(5/5),Cap_R0.01_H0L0_T0.1_A0.2_Q(1/5),Cap_R0.01_H0L0_T0.1_A0.2_Q(2/5),Cap_R0.01_H0L0_T0.1_A0.2_Q(3/5),Cap_R0.01_H0L0_T0.1_A0.2_Q(4/5),Cap_R0.01_H0L0_T0.1_A0.2_Q(5/5)
2025-06-30,-0.045390,-0.061082,-0.060194,-0.080546,-0.050558,-0.041202,-0.055873,-0.054596,-0.075297,-0.045946,-0.087058,-0.090291,-0.200019,-0.093525,-0.179529,-0.082571,-0.086592,-0.196133,-0.089515,-0.175322
2025-07-31,-0.023661,-0.034123,-0.027210,-0.019895,0.017842,-0.019439,-0.028394,-0.021778,-0.014544,0.022970,-0.080010,-0.095493,-0.033371,-0.058410,0.009097,-0.076022,-0.092756,-0.028624,-0.054544,0.012974
2025-08-31,0.011416,0.015842,0.008370,0.011976,0.017243,0.015539,0.021596,0.014369,0.017443,0.021852,0.018383,0.010417,0.010611,0.012448,0.008534,0.023711,0.016066,0.016276,0.016997,0.013128
2025-09-30,-0.005521,-0.026777,-0.038334,-0.047139,-0.037890,-0.001698,-0.021503,-0.032704,-0.042104,-0.033668,-0.019315,-0.135767,-0.042787,-0.109166,-0.027861,-0.013674,-0.131218,-0.037619,-0.105237,-0.023876
2025-10-31,0.001670,-0.011543,-0.013299,-0.023562,-0.025665,0.005539,-0.005824,-0.007333,-0.018363,-0.021266,-0.110482,-0.269906,-0.139306,-0.134752,-0.071433,-0.108576,-0.267635,-0.136065,-0.131327,-0.066688


In [112]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

subset = select_columns(
    portfolio_df_2,
    {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

In [113]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 데이터 준비
nav = (subset + 1).cumprod()
log_cum = np.log1p(subset).cumsum()

# 서브플롯 (가로 2개)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Cumulative NAV", "Log Cumulative Return"])

# 첫 번째 그래프
for col in nav.columns:
    fig.add_trace(
        go.Scatter(x=nav.index, y=nav[col], mode='lines', name=col,
                   legendgroup=col, showlegend=True),
        row=1, col=1
    )

# 두 번째 그래프
for col in log_cum.columns:
    fig.add_trace(
        go.Scatter(x=log_cum.index, y=log_cum[col], mode='lines', name=col,
                   legendgroup=col, showlegend=True),
        row=1, col=2
    )

# 레이아웃 기본 설정
fig.update_layout(
    title="Portfolio Performance Comparison",
    height=500, width=1000,
    template="plotly_white"
)

fig.update_layout(
    shapes=[
        # 왼쪽 그래프 영역 (x=0~0.45, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.0, y0=0.0, x1=0.45, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
        
        # 오른쪽 그래프 영역 (x=0.55~1, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.55, y0=0.0, x1=1.0, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
    ]
)

##### 롱숏 포트폴리오

In [126]:
def run_long_short_strategy(wins_threshold_temp, weight_method_temp, cost_temp,
                            trading_threshold_temp, Amihud_threshold_temp, 
                            initial_NAV=1):
    
    high_cost, low_cost = cost_temp
    NAV = initial_NAV
    
    portfolio_name = f"LongShort_R{wins_threshold_temp}_H{int(high_cost*10000)}L{int(low_cost*10000)}_T{trading_threshold_temp}_A{Amihud_threshold_temp}"
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade = pd.Series(dtype=float, name=portfolio_name)
    
    # 수익률 윈저라이징
    ret_wins = ret_m.clip(
        lower = ret_m.quantile(wins_threshold_temp),
        upper = ret_m.quantile(1 - wins_threshold_temp),
        axis  = 1
    )
    
    prev_portfolio_long = None
    prev_portfolio_short = None
    
    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]
        
        # 거래대금 하위 x% 필터링
        series = trading_value_60.loc[start_date]
        threshold = series.quantile(trading_threshold_temp)
        filtered = series[series > threshold]
        factor_filtered = factor_df.loc[start_date, filtered.index]
        
        # 분위수 계산 (10분위)
        quantiles = pd.qcut(factor_filtered, q=10, labels=False, duplicates='drop')
        
        # 롱: 하위 10% (quantile 0)
        long_basket = factor_filtered[quantiles == 0]
        # 숏: 상위 10% (quantile 9)
        short_basket = factor_filtered[quantiles == 9]
        
        if long_basket.empty or short_basket.empty:
            continue
        
        # Amihud illiquidity 계산
        illiq = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
        threshold = illiq.quantile(1 - Amihud_threshold_temp)
        illiquid_top = illiq[illiq >= threshold].index
        
        # === 롱 포트폴리오 ===
        if prev_portfolio_long is None:
            prev_weights_long = pd.Series(0, index=long_basket.index)
        else:
            prev_weights_long = prev_portfolio_long / prev_portfolio_long.sum()
        
        if weight_method_temp == 'Equal':
            target_weights_long = pd.Series(1/len(long_basket), index=long_basket.index)
        else:  # Cap
            cap_seg_long = mkt_cap.loc[start_date, long_basket.index]
            target_weights_long = cap_seg_long / cap_seg_long.sum()
        
        all_index_long = target_weights_long.index.union(prev_weights_long.index)
        target_w_long = target_weights_long.reindex(all_index_long, fill_value=0)
        prev_w_long = prev_weights_long.reindex(all_index_long, fill_value=0)
        delta_w_long = target_w_long - prev_w_long
        
        trade_amounts_long = abs(delta_w_long) * NAV / 2  # 롱-숏 각각 50% 배분
        cost_rate_long = np.where(delta_w_long.index.isin(illiquid_top), high_cost, low_cost)
        trade_cost_long = (trade_amounts_long * cost_rate_long).sum()
        
        current_portfolio_value_long = target_weights_long * (NAV / 2 - trade_cost_long)
        ret_seg_long = ret_wins.loc[end_date, long_basket.index]
        next_portfolio_value_long = current_portfolio_value_long * (ret_seg_long + 1)
        
        # === 숏 포트폴리오 ===
        if prev_portfolio_short is None:
            prev_weights_short = pd.Series(0, index=short_basket.index)
        else:
            prev_weights_short = prev_portfolio_short / prev_portfolio_short.sum()
        
        if weight_method_temp == 'Equal':
            target_weights_short = pd.Series(1/len(short_basket), index=short_basket.index)
        else:  # Cap
            cap_seg_short = mkt_cap.loc[start_date, short_basket.index]
            target_weights_short = cap_seg_short / cap_seg_short.sum()
        
        all_index_short = target_weights_short.index.union(prev_weights_short.index)
        target_w_short = target_weights_short.reindex(all_index_short, fill_value=0)
        prev_w_short = prev_weights_short.reindex(all_index_short, fill_value=0)
        delta_w_short = target_w_short - prev_w_short
        
        trade_amounts_short = abs(delta_w_short) * NAV / 2  # 롱-숏 각각 50% 배분
        cost_rate_short = np.where(delta_w_short.index.isin(illiquid_top), high_cost, low_cost)
        trade_cost_short = (trade_amounts_short * cost_rate_short).sum()
        
        current_portfolio_value_short = target_weights_short * (NAV / 2 - trade_cost_short)
        ret_seg_short = ret_wins.loc[end_date, short_basket.index]
        next_portfolio_value_short = current_portfolio_value_short * (1 - ret_seg_short)  # 숏: 수익률 반전
        
        # === 롱-숏 포트폴리오 합산 ===
        NAV_new_long = next_portfolio_value_long.sum()
        NAV_new_short = next_portfolio_value_short.sum()
        NAV_new = NAV_new_long + NAV_new_short
        
        portfolio_ret = NAV_new / NAV - 1
        
        prev_portfolio_long = next_portfolio_value_long
        prev_portfolio_short = next_portfolio_value_short
        NAV = NAV_new
        
        total_trade.loc[start_date] = trade_amounts_long.sum() + trade_amounts_short.sum()
        portfolio_return.loc[end_date] = portfolio_ret
    
    return portfolio_return, total_trade

# 포트폴리오 선택 함수
def select_columns(df, fixed_options):
    selected = []
    new_names = {}
    
    for c in df.columns:
        parts = c.split('_')
        
        # 조건 체크 (고정한 옵션은 정확히 일치해야 함)
        if all(parts[i] == v or v == "*" for i, v in fixed_options.items()):
            selected.append(c)
            
            # 새 이름은 "고정하지 않은 옵션만"
            free_parts = [parts[i] for i, v in fixed_options.items() if v == "*"]
            new_name = "_".join(free_parts)
            new_names[c] = new_name
    
    # 선택된 subset 반환, 컬럼명은 자유 요소만
    subset = df[selected].rename(columns=new_names)
    return subset

In [127]:
# 팩터 설정
factor_df         = MAX5_df

In [128]:
start_point       = '2000-01-01'                     # 백테스트 기간 설정
end_point         = '2025-10-31'

weight_method     = ['Equal']                 # ['Equal', Cap']

cost              = [(0.008, 0.003), (0, 0)]         # (high_cost, low_cost)            e.g. [(0.008, 0.003), (0, 0)]
                                                     
wins_threshold    = [0.01]                           # 수익률 윈저라이징                  e.g. [0.01, 0.05]
trading_threshold = [0.1]                            # 60일 평균 거래대금 필터링           e.g. [0.1, 0.0]
Amihud_threshold  = [0.2]                            # 유동성 기준                       e.g. [0.2, 0.0]

initial_NAV       = 1                                # 초기값

In [129]:
# ret_m        = total_adj_close.resample('QE').last().ffill().pct_change() # 분기 리밸런싱
ret_m        = total_adj_close.resample('ME').last().ffill().pct_change()

portfolio_df_3 = pd.DataFrame()
trade_df_3     = pd.DataFrame()

month_ends   = factor_df.resample('ME').last().index
month_ends   = month_ends[(month_ends >= pd.to_datetime(start_point)) &
                        (month_ends <= pd.to_datetime(end_point))]

C:\Users\XH58\AppData\Local\Temp\ipykernel_11812\869844599.py:2: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



In [130]:
# 모든 조합 만들기
param_list = [
    (wins_threshold_temp, weight_method_temp, cost_temp, trading_threshold_temp, Amihud_threshold_temp, n_temp)
    for wins_threshold_temp in wins_threshold
    for weight_method_temp in weight_method
    for cost_temp in cost
    for trading_threshold_temp in trading_threshold
    for Amihud_threshold_temp in Amihud_threshold
    for n_temp in n
]

# 병렬 실행
results = Parallel(n_jobs=-1)(
    delayed(run_long_short_strategy)(*params) for params in tqdm(param_list)
)

# 결과 합치기
portfolio_df_3 = pd.DataFrame({r[0].name: r[0] for r in results})
trade_df_3     = pd.DataFrame({r[1].name: r[1] for r in results})

100%|██████████| 10/10 [00:00<00:00, 5002.75it/s]


In [131]:
portfolio_df_3.tail()

,LongShort_R0.01_H80L30_T0.1_A0.2,LongShort_R0.01_H0L0_T0.1_A0.2
2025-06-30,0.008269,0.013622
2025-07-31,0.023704,0.029445
2025-08-31,-0.000971,0.003921
2025-09-30,-0.016046,-0.011243
2025-10-31,-0.032038,-0.027162


In [132]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

subset = select_columns(
    portfolio_df_3,
    {0: "LongShort", 1: "R0.01", 2: "*", 3: "T0.1", 4:"A0.2"}
)

In [133]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 데이터 준비
nav = (subset + 1).cumprod()
log_cum = np.log1p(subset).cumsum()

# 서브플롯 (가로 2개)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Cumulative NAV", "Log Cumulative Return"])

# 첫 번째 그래프
for col in nav.columns:
    fig.add_trace(
        go.Scatter(x=nav.index, y=nav[col], mode='lines', name=col,
                   legendgroup=col, showlegend=True),
        row=1, col=1
    )

# 두 번째 그래프
for col in log_cum.columns:
    fig.add_trace(
        go.Scatter(x=log_cum.index, y=log_cum[col], mode='lines', name=col,
                   legendgroup=col, showlegend=True),
        row=1, col=2
    )

# 레이아웃 기본 설정
fig.update_layout(
    title="Portfolio Performance Comparison",
    height=500, width=1000,
    template="plotly_white"
)

fig.update_layout(
    shapes=[
        # 왼쪽 그래프 영역 (x=0~0.45, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.0, y0=0.0, x1=0.45, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
        
        # 오른쪽 그래프 영역 (x=0.55~1, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.55, y0=0.0, x1=1.0, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
    ]
)

In [134]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

# 동일가중 포트폴리오
subset_cost = select_columns(
    portfolio_df_3,
    {0: "LongShort", 1: "R0.01", 2: "*", 3: "T0.1", 4:"A0.2"}
)

In [135]:
import numpy as np
import plotly.graph_objects as go

# 로그누적수익률 계산
log_cum_cost = np.log1p(subset_cost).cumsum()

# 단일 plot 생성
fig = go.Figure()

for col in log_cum_cost.columns:
    fig.add_trace(
        go.Scatter(
            x=log_cum_cost.index, 
            y=log_cum_cost[col], 
            mode='lines', 
            name=col
        )
    )

# 레이아웃 설정
fig.update_layout(
    title="Equal-weighted Portfolios (Cost Parameter Variation, Log Cumulative Return)",
    xaxis_title="Date",
    yaxis_title="Log Cumulative Return",
    template="plotly_white",
    height=600,
    width=900
)

In [136]:
import numpy as np

# 포트폴리오 수익률 추출
portfolio_returns = portfolio_df_3['LongShort_R0.01_H80L30_T0.1_A0.2'].dropna()

if len(portfolio_returns) > 0:
    # 월별 수익률 기준 계산
    monthly_returns = portfolio_returns
    
    # 총 기간 (년)
    total_months = len(monthly_returns)
    total_years = total_months / 12
    
    # NAV 계산
    nav = (1 + monthly_returns).cumprod()
    
    # CAGR (연평균 수익률)
    total_return = nav.iloc[-1] / nav.iloc[0] - 1
    cagr = (1 + total_return) ** (1 / total_years) - 1 if total_years > 0 else np.nan
    
    # 연화 표준편차 (월별 표준편차 * sqrt(12))
    monthly_std = monthly_returns.std()
    annual_volatility = monthly_std * np.sqrt(12)
    
    # 샤프 비율 (무위험 수익률 0 가정)
    sharpe_ratio = cagr / annual_volatility if annual_volatility != 0 else np.nan
    
    print(f"포트폴리오: LongShort_R0.01_H80L30_T0.1_A0.2")
    print(f"CAGR (연평균 수익률): {cagr:.4f} ({cagr*100:.2f}%)")
    print(f"연화 표준편차: {annual_volatility:.4f} ({annual_volatility*100:.2f}%)")
    print(f"샤프 비율: {sharpe_ratio:.4f}")
else:
    print("데이터가 없습니다.")

print()

# 포트폴리오 수익률 추출
portfolio_returns = portfolio_df_3['LongShort_R0.01_H0L0_T0.1_A0.2'].dropna()

if len(portfolio_returns) > 0:
    # 월별 수익률 기준 계산
    monthly_returns = portfolio_returns
    
    # 총 기간 (년)
    total_months = len(monthly_returns)
    total_years = total_months / 12
    
    # NAV 계산
    nav = (1 + monthly_returns).cumprod()
    
    # CAGR (연평균 수익률)
    total_return = nav.iloc[-1] / nav.iloc[0] - 1
    cagr = (1 + total_return) ** (1 / total_years) - 1 if total_years > 0 else np.nan
    
    # 연화 표준편차 (월별 표준편차 * sqrt(12))
    monthly_std = monthly_returns.std()
    annual_volatility = monthly_std * np.sqrt(12)
    
    # 샤프 비율 (무위험 수익률 0 가정)
    sharpe_ratio = cagr / annual_volatility if annual_volatility != 0 else np.nan
    
    print(f"포트폴리오: LongShort_R0.01_H0L0_T0.1_A0.2")
    print(f"CAGR (연평균 수익률): {cagr:.4f} ({cagr*100:.2f}%)")
    print(f"연화 표준편차: {annual_volatility:.4f} ({annual_volatility*100:.2f}%)")
    print(f"샤프 비율: {sharpe_ratio:.4f}")
else:
    print("데이터가 없습니다.")

포트폴리오: LongShort_R0.01_H80L30_T0.1_A0.2
CAGR (연평균 수익률): 0.1457 (14.57%)
연화 표준편차: 0.1009 (10.09%)
샤프 비율: 1.4447

포트폴리오: LongShort_R0.01_H0L0_T0.1_A0.2
CAGR (연평균 수익률): 0.2195 (21.95%)
연화 표준편차: 0.1014 (10.14%)
샤프 비율: 2.1639


In [137]:
# portfolio_df_3를 CSV 파일로 저장
portfolio_df_3.to_csv('./output/FF3/LongShort_portfoilo_return.csv', index=True, encoding='utf-8-sig')
print("CSV 파일 저장 완료: ./output/FF3/LongShort_portfoilo_return.csv")

CSV 파일 저장 완료: ./output/FF3/LongShort_portfoilo_return.csv
